# Thực nghiệm weakly supervised Learning-to-Rank cho CV–Job Matching tiếng Việt

Notebook này là **tài liệu thực nghiệm chính, độc lập và có thể thực thi từ đầu đến cuối**. Toàn bộ mã nguồn cần thiết—từ audit dữ liệu, xây dựng ba tín hiệu, mô hình nhãn Dawid–Skene, ba hàm mục tiêu LTR, lựa chọn mô hình, khóa giao thức, đánh giá Gold-test đến paired bootstrap—được định nghĩa trực tiếp trong notebook; notebook không gọi logic thực nghiệm từ `src/`.

## Câu hỏi nghiên cứu

- **RQ chính:** Hàm xếp hạng học từ nhãn yếu có xếp hạng tốt hơn công thức điểm thủ công trên Gold-test độc lập hay không?
- **RQ1:** Xác suất ước lượng bởi mô hình nhãn có đạt điều kiện chất lượng đã khai báo trước trên Gold-validation hay không?
- **RQ2:** Pointwise, pairwise hay listwise phù hợp nhất khi giữ nguyên tập feature và năng lực scorer?

## Giao thức xác nhận

1. Mô hình đề xuất chỉ nhận ba tín hiệu: semantic, skill và experience.
2. Mọi phép fit preprocessing, ngưỡng labeling function và Dawid–Skene chỉ dùng development-train.
3. Gold được tách query-disjoint: số job validation khóa trong cấu hình, phần còn lại là Gold-test.
4. Gold-validation chỉ dùng cho ngưỡng posterior và lựa chọn formulation; không dùng Gold-test để tuning.
5. Gold-test chỉ được transform và đánh giá sau khi protocol đã khóa, đúng một lần trong một kernel sạch.
6. Đơn vị thống kê là job; metric được tính từng job rồi macro-average và bootstrap theo job.
7. Chỉ có hai ablation: thay label model bằng trung bình ba tín hiệu; bỏ LTR và xếp hạng trực tiếp bằng xác suất ước lượng.
8. Xác suất Dawid–Skene là **nhãn yếu ước lượng**, không phải ground truth. Gold hiện có là benchmark do tác giả gán nhãn, không phải outcome tuyển dụng.


## 1. Môi trường, khả năng tái lập và cấu hình khóa trước

Cell dưới đây tải thư viện, xác định thư mục dự án và đọc cấu hình full. Notebook **không có chế độ smoke**. Các seed, kích thước mẫu, hyperparameter grid, ngưỡng LF và số bootstrap được hiển thị trước khi xử lý Gold-test.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import importlib.metadata
import itertools
import json
import platform
import random
import re
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from torch import nn

def find_project_root(start: str | Path) -> Path:
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "experiment_3signal.yaml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find configs/experiment_3signal.yaml from the current directory"
    )


ROOT = find_project_root(Path.cwd())
CONFIG_PATH = ROOT / "configs" / "experiment_3signal.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = (ROOT / CONFIG["data_dir"]).resolve()
GOLD_PATH = (ROOT / CONFIG["gold_path"]).resolve()
ANNOTATION_VALUE = CONFIG["gold"].get("independent_annotations_path")
INDEPENDENT_ANNOTATIONS_PATH = (
    (ROOT / ANNOTATION_VALUE).resolve() if ANNOTATION_VALUE else None
)

assert CONFIG["seeds"] == [11, 23, 42, 67, 89]
assert CONFIG["gold"]["validation_jobs"] == 4
assert CONFIG["gold"]["k_values"] == [5, 10]
assert CONFIG["sample"] == {
    "n_jobs": 2000, "n_candidates": 1500, "candidates_per_job": 200,
}
assert CONFIG["models"]["weight_decay"] == 1e-4
assert CONFIG["models"]["gradient_clip_norm"] == 5.0
REQUESTED_DEVICE = str(CONFIG["models"]["device"])
if REQUESTED_DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "The configured CUDA device is unavailable; the active PyTorch build "
        "does not expose an NVIDIA GPU"
    )
DEVICE = torch.device("cuda:0" if REQUESTED_DEVICE == "cuda" else REQUESTED_DEVICE)
assert CONFIG["weak_supervision"]["posterior_threshold"] == 0.5
assert CONFIG["weak_supervision"]["label_model"] == {
    "n_iter": 100,
    "parameter_clip": [0.51, 0.99],
    "prior_clip": [0.05, 0.95],
    "initial_accuracy": 0.75,
    "convergence_tolerance": 1e-7,
}
assert set(CONFIG["models"]["batch_sizes"]) == {
    "pointwise", "pairwise", "listwise",
}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
print("Project root:", ROOT)
print("Real data root:", DATA_ROOT)
print("Gold path:", GOLD_PATH)
print("Protocol version:", CONFIG["protocol_version"])
print("Configured device:", DEVICE)
if DEVICE.type == "cuda":
    print("CUDA device:", torch.cuda.get_device_name(DEVICE))
    print("CUDA runtime:", torch.version.cuda)
display(pd.DataFrame({
    "setting": ["seeds", "sample", "learning_rates", "batch_sizes", "epochs", "bootstrap_resamples"],
    "value": [
        str(CONFIG["seeds"]), str(CONFIG["sample"]),
        str(CONFIG["models"]["learning_rates"]), str(CONFIG["models"]["batch_sizes"]),
        CONFIG["models"]["max_epochs"], CONFIG["bootstrap"]["n_resamples"],
    ],
}))


## 2. Hằng số và tiện ích tái lập

Các hàm băm, ghi artifact, chuẩn hóa Unicode và seed xác định được định nghĩa tại đây.


In [ ]:
FEATURE_COLUMNS = ["s_sem", "s_skill", "s_exp"]
LF_COLUMNS = ["lf_sem", "lf_skill", "lf_exp"]


In [ ]:
def set_seed(seed: int) -> np.random.RandomState:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    return np.random.RandomState(seed)


In [ ]:
def normalize_text(value) -> str:
    text = unicodedata.normalize("NFC", str(value or "")).strip().lower()
    if text in {"", "nan", "none", "null", "n/a", "na", "chưa cập nhật"}:
        return ""
    return " ".join(text.split())


In [ ]:
def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


In [ ]:
def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
def write_json(path: str | Path, value) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=_json_default), encoding="utf-8")


In [ ]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)


## 3. Nạp dữ liệu, kiểm tra identity và chia query

Identity ổn định được ánh xạ theo chỉ số dòng gốc (`JOB_<index>`, `CV_<index>`). CV trùng hoàn toàn được loại nhưng `_source_index` được bảo toàn. Manifest Gold dùng số job validation trong cấu hình, danh sách `job_id` đã sort và seed cố định; tuyệt đối không đọc grade/relevance khi chia split. IAA chỉ được tính từ file nhãn độc lập thật.


In [ ]:
JOB_REQUIRED_COLUMNS = [
    "JobID", "URL Job", "Job Title", "Job Description", "Job Requirements",
    "Job Address", "Years of Experience",
]


In [ ]:
CANDIDATE_REQUIRED_COLUMNS = [
    "URL User", "UserID", "Desired Job", "Workplace Desired", "Target",
    "Skills", "Work Experience",
]


In [ ]:
RAW_FILENAMES = {
    "jobs": "JOB_DATA_FINAL.csv",
    "candidates": "USER_DATA_FINAL.csv",
}


In [ ]:
def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
def resolve_data_paths(data_dir: str | Path) -> dict[str, Path]:
    root = Path(data_dir).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Real data directory does not exist: {root}")
    paths = {name: root / filename for name, filename in RAW_FILENAMES.items()}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Required real-data files are missing: " + ", ".join(missing)
        )
    return paths


In [ ]:
def build_input_manifest(data_dir: str | Path) -> dict[str, dict]:
    paths = resolve_data_paths(data_dir)
    manifest = {}
    for name, path in paths.items():
        stat = path.stat()
        manifest[name] = {
            "path": str(path),
            "size_bytes": int(stat.st_size),
            "modified_at_utc": datetime.fromtimestamp(
                stat.st_mtime, tz=timezone.utc
            ).isoformat(),
            "sha256": _sha256_file(path),
        }
    return manifest


In [ ]:
def format_job_id(index: int) -> str:
    return f"JOB_{int(index)}"


In [ ]:
def format_candidate_id(index: int) -> str:
    return f"CV_{int(index):03d}"


In [ ]:
def parse_entity_index(value: str) -> int:
    match = re.search(r"(\d+)$", str(value))
    if not match:
        raise ValueError(f"Invalid entity identifier: {value}")
    return int(match.group(1))


In [ ]:
@dataclass
class RawData:
    jobs: pd.DataFrame
    candidates: pd.DataFrame
    audit: dict


In [ ]:
@dataclass
class SampledData:
    jobs: pd.DataFrame
    candidates: pd.DataFrame
    pairs: pd.DataFrame


In [ ]:
def load_raw_data(data_dir: str | Path) -> RawData:
    paths = resolve_data_paths(data_dir)
    root = Path(data_dir).expanduser().resolve()
    input_files = build_input_manifest(root)
    jobs_raw = pd.read_csv(paths["jobs"])
    candidates_raw = pd.read_csv(paths["candidates"])
    missing_jobs = set(JOB_REQUIRED_COLUMNS) - set(jobs_raw.columns)
    missing_candidates = set(CANDIDATE_REQUIRED_COLUMNS) - set(candidates_raw.columns)
    if missing_jobs or missing_candidates:
        raise ValueError(
            f"Missing raw columns: jobs={sorted(missing_jobs)}, "
            f"candidates={sorted(missing_candidates)}"
        )

    duplicate_candidates = int(candidates_raw.duplicated().sum())
    jobs = jobs_raw.assign(_source_index=jobs_raw.index).copy()
    candidates = (
        candidates_raw.assign(_source_index=candidates_raw.index)
        .drop_duplicates(subset=[column for column in candidates_raw.columns], keep="first")
        .reset_index(drop=True)
    )
    if jobs["JobID"].duplicated().any() or jobs["URL Job"].duplicated().any():
        raise ValueError("Job identifiers or URLs are not unique")
    if candidates["UserID"].duplicated().any() or candidates["URL User"].duplicated().any():
        raise ValueError("Candidate identifiers or URLs remain duplicated after deduplication")

    audit = {
        "data_root": str(root),
        "input_files": input_files,
        "jobs_raw_rows": int(len(jobs_raw)),
        "jobs_clean_rows": int(len(jobs)),
        "jobs_columns": int(jobs_raw.shape[1]),
        "jobs_unique": int(jobs["JobID"].nunique()),
        "jobs_exact_duplicates": int(jobs_raw.duplicated().sum()),
        "jobs_missing_cells": int(jobs_raw.isna().sum().sum()),
        "jobs_required_columns": list(JOB_REQUIRED_COLUMNS),
        "candidates_raw_rows": int(len(candidates_raw)),
        "candidates_clean_rows": int(len(candidates)),
        "candidates_columns": int(candidates_raw.shape[1]),
        "candidates_unique": int(candidates["UserID"].nunique()),
        "candidates_exact_duplicates": duplicate_candidates,
        "candidates_missing_cells": int(candidates_raw.isna().sum().sum()),
        "candidates_required_columns": list(CANDIDATE_REQUIRED_COLUMNS),
    }
    return RawData(jobs=jobs, candidates=candidates, audit=audit)


In [ ]:
def load_gold_with_identity_check(
    gold_path: str | Path,
    raw: RawData,
) -> pd.DataFrame:
    gold = pd.read_csv(gold_path)
    required = {"job_id", "cand_id", "job_title", "desired_job", "relevance"}
    missing = required - set(gold.columns)
    if missing:
        raise ValueError(f"Gold columns missing: {sorted(missing)}")
    job_indices = gold["job_id"].map(parse_entity_index)
    candidate_indices = gold["cand_id"].map(parse_entity_index)
    jobs_by_source = raw.jobs.set_index("_source_index")
    candidates_by_source = raw.candidates.set_index("_source_index")
    if not set(job_indices).issubset(jobs_by_source.index):
        raise ValueError("Gold contains an unknown raw job index")
    if not set(candidate_indices).issubset(candidates_by_source.index):
        raise ValueError("Gold contains an unknown or deduplicated candidate index")
    expected_jobs = job_indices.map(jobs_by_source["Job Title"]).fillna("").str.strip()
    expected_candidates = candidate_indices.map(candidates_by_source["Desired Job"]).fillna("").str.strip()
    if not expected_jobs.equals(gold["job_title"].fillna("").str.strip()):
        raise ValueError("Gold job IDs do not match raw job titles")
    if not expected_candidates.equals(gold["desired_job"].fillna("").str.strip()):
        raise ValueError("Gold candidate IDs do not match raw desired jobs")
    if not gold["relevance"].isin([0, 1, 2, 3]).all():
        raise ValueError("Gold relevance must be in {0,1,2,3}")
    if gold.duplicated(["job_id", "cand_id"]).any():
        raise ValueError("Gold contains duplicate CV--job pairs")
    return gold.reset_index(drop=True)


In [ ]:
def make_gold_split_manifest(
    gold: pd.DataFrame,
    n_validation_jobs: int = 4,
    seed: int = 42,
) -> dict:
    jobs = sorted(gold["job_id"].unique())
    if len(jobs) < 2:
        raise ValueError("Gold requires at least two jobs for query-disjoint validation/test")
    if not 1 <= n_validation_jobs < len(jobs):
        raise ValueError(
            "n_validation_jobs must leave at least one Gold-test job"
        )
    shuffled_jobs = np.asarray(jobs, dtype=object)
    rng = np.random.RandomState(seed)
    rng.shuffle(shuffled_jobs)
    validation_jobs = sorted(shuffled_jobs[:n_validation_jobs].tolist())
    test_jobs = sorted(set(jobs) - set(validation_jobs))
    manifest = {
        "protocol": "query-disjoint-gold-label-blind-v3",
        "split_algorithm": "sorted-job-id-seeded-shuffle-v1",
        "seed": int(seed),
        "total_jobs": len(jobs),
        "validation_jobs": validation_jobs,
        "test_jobs": test_jobs,
        "validation_pairs": int(gold["job_id"].isin(validation_jobs).sum()),
        "test_pairs": int(gold["job_id"].isin(test_jobs).sum()),
    }
    if set(validation_jobs) & set(test_jobs):
        raise AssertionError("Gold validation/test jobs overlap")
    return manifest


In [ ]:
def inter_annotator_agreement(
    annotation_path: str | Path | None,
    gold: pd.DataFrame,
) -> dict:
    if annotation_path is None:
        return {
            "status": "not_available",
            "reason": "No independent annotation file configured",
        }
    path = Path(annotation_path).expanduser().resolve()
    if not path.is_file():
        return {
            "status": "not_available",
            "reason": f"Independent annotation file not found: {path}",
        }

    annotations = pd.read_csv(path)
    required = {"job_id", "cand_id", "annotator_id", "relevance"}
    missing = required - set(annotations.columns)
    if missing:
        raise ValueError(f"Annotation columns missing: {sorted(missing)}")
    annotations = annotations[list(required)].copy()
    annotations["annotator_id"] = annotations["annotator_id"].astype(str).str.strip()
    if annotations["annotator_id"].eq("").any():
        raise ValueError("annotator_id must not be empty")
    if not annotations["relevance"].isin([0, 1, 2, 3]).all():
        raise ValueError("Annotation relevance must be in {0,1,2,3}")
    if annotations.duplicated(["job_id", "cand_id", "annotator_id"]).any():
        raise ValueError("Duplicate annotation for an annotator and CV--job pair")

    gold_pairs = set(map(tuple, gold[["job_id", "cand_id"]].itertuples(index=False, name=None)))
    annotation_pairs = set(map(tuple, annotations[["job_id", "cand_id"]].itertuples(index=False, name=None)))
    unknown = sorted(annotation_pairs - gold_pairs)
    if unknown:
        raise ValueError(f"Annotations contain pairs outside Gold: {unknown[:3]}")
    annotators = sorted(annotations["annotator_id"].unique())
    if len(annotators) != 2:
        raise ValueError("IAA requires exactly two independent annotators")

    paired = annotations.pivot(
        index=["job_id", "cand_id"], columns="annotator_id", values="relevance"
    ).dropna()
    if paired.empty:
        raise ValueError("Annotators have no overlapping CV--job pairs")
    left = paired[annotators[0]].astype(int)
    right = paired[annotators[1]].astype(int)
    return {
        "status": "available",
        "annotation_path": str(path),
        "annotators": annotators,
        "overlap_pairs": int(len(paired)),
        "gold_pairs": int(len(gold)),
        "coverage": float(len(paired) / len(gold)),
        "exact_agreement": float((left == right).mean()),
        "quadratic_weighted_kappa": float(
            cohen_kappa_score(left, right, labels=[0, 1, 2, 3], weights="quadratic")
        ),
    }


In [ ]:
def sample_development_entities(
    raw: RawData,
    n_jobs: int,
    n_candidates: int,
    candidates_per_job: int,
    seed: int,
    excluded_job_ids: set[str],
    excluded_candidate_ids: set[str],
) -> SampledData:
    excluded_job_indices = {parse_entity_index(value) for value in excluded_job_ids}
    excluded_candidate_indices = {parse_entity_index(value) for value in excluded_candidate_ids}
    jobs = raw.jobs[~raw.jobs["_source_index"].isin(excluded_job_indices)]
    candidates = raw.candidates[~raw.candidates["_source_index"].isin(excluded_candidate_indices)]
    if n_jobs > len(jobs) or n_candidates > len(candidates):
        raise ValueError("Requested sample exceeds non-Gold entities")
    jobs = jobs.sample(n=n_jobs, random_state=seed).reset_index(drop=True)
    candidates = candidates.sample(n=n_candidates, random_state=seed).reset_index(drop=True)
    rng = np.random.RandomState(seed)
    rows = []
    for _, job in jobs.iterrows():
        selected = rng.choice(
            len(candidates),
            size=min(candidates_per_job, len(candidates)),
            replace=False,
        )
        for candidate_position in selected:
            candidate = candidates.iloc[int(candidate_position)]
            rows.append({
                "pair_id": len(rows),
                "job_id": format_job_id(job["_source_index"]),
                "cand_id": format_candidate_id(candidate["_source_index"]),
                "job_source_index": int(job["_source_index"]),
                "candidate_source_index": int(candidate["_source_index"]),
            })
    pairs = pd.DataFrame(rows)
    if pairs.duplicated(["job_id", "cand_id"]).any():
        raise AssertionError("Duplicate development pair")
    return SampledData(jobs=jobs, candidates=candidates, pairs=pairs)


In [ ]:
def split_development_pairs(
    pairs: pd.DataFrame,
    train_ratio: float,
    validation_ratio: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    jobs = np.array(sorted(pairs["job_id"].unique()))
    rng = np.random.RandomState(seed)
    rng.shuffle(jobs)
    n_train = int(len(jobs) * train_ratio)
    n_validation = int(len(jobs) * validation_ratio)
    train_jobs = set(jobs[:n_train])
    validation_jobs = set(jobs[n_train:n_train + n_validation])
    test_jobs = set(jobs[n_train + n_validation:])
    if not train_jobs or not validation_jobs or not test_jobs:
        raise ValueError("Development split produced an empty partition")
    if train_jobs & validation_jobs or train_jobs & test_jobs or validation_jobs & test_jobs:
        raise AssertionError("Development jobs overlap")

    def select(values: set[str]) -> pd.DataFrame:
        return pairs[pairs["job_id"].isin(values)].copy().reset_index(drop=True)

    manifest = {
        "train_jobs": sorted(train_jobs),
        "validation_jobs": sorted(validation_jobs),
        "test_jobs": sorted(test_jobs),
    }
    return select(train_jobs), select(validation_jobs), select(test_jobs), manifest


In [ ]:
def gold_pairs_to_raw_indices(gold: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "pair_id": gold.get("pair_id", pd.Series(range(len(gold)))),
        "job_id": gold["job_id"].to_numpy(),
        "cand_id": gold["cand_id"].to_numpy(),
        "job_source_index": gold["job_id"].map(parse_entity_index).to_numpy(int),
        "candidate_source_index": gold["cand_id"].map(parse_entity_index).to_numpy(int),
    })


## 4. Xây dựng ba tín hiệu và baseline thủ công

- `s_sem`: cosine giữa multilingual sentence embeddings của `Job Title` + `Job Description` + `Job Requirements` và `Desired Job` + `Target` + `Skills`.
- `s_skill`: Jaccard giữa tập kỹ năng đã chuẩn hóa.
- `s_exp`: độ tương thích giữa hai khoảng kinh nghiệm.

Model embedding pretrained được khóa trong cấu hình và không fine-tune bằng Gold. TF–IDF chỉ fit trên development-train cho role/description lexical components của baseline. Scorer học máy chỉ nhận đúng ba cột trên.


In [ ]:
_SKILL_ALIASES = {
    "python3": "python", "python 3": "python", "py": "python",
    "machine-learning": "machine learning", "ml": "machine learning",
    "deep-learning": "deep learning", "dl": "deep learning",
    "powerbi": "power bi", "power-bi": "power bi",
    "nodejs": "node.js", "postgresql": "postgres", "sql server": "sql",
    "ms office": "microsoft office", "tin học văn phòng": "microsoft office",
}


In [ ]:
_KNOWN_SKILLS = sorted(set(_SKILL_ALIASES.values()) | {
    "python", "java", "javascript", "typescript", "c++", "c#", "php", "sql",
    "excel", "microsoft office", "power bi", "tableau", "machine learning",
    "deep learning", "data analysis", "data science", "computer vision",
    "natural language processing", "project management", "sales", "marketing",
    "seo", "accounting", "logistics", "human resources", "customer service",
    "communication", "teamwork", "leadership", "english", "chinese", "japanese",
}, key=len, reverse=True)


In [ ]:
def normalize_location(value) -> str:
    text = normalize_text(value)
    replacements = {
        "tp.hcm": "hồ chí minh", "tphcm": "hồ chí minh", "tp hcm": "hồ chí minh",
        "hcm": "hồ chí minh", "hn": "hà nội",
    }
    for source, target in replacements.items():
        text = text.replace(source, target)
    return text


In [ ]:
def normalize_skill(value: str) -> str:
    skill = normalize_text(value)
    skill = re.sub(r"^[\W_]+|[\W_]+$", "", skill)
    skill = re.sub(r"\s+", " ", skill)
    return _SKILL_ALIASES.get(skill, skill)


In [ ]:
def extract_skill_set(value) -> set[str]:
    text = normalize_text(value)
    if not text:
        return set()
    found = {
        phrase for phrase in _KNOWN_SKILLS
        if re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text)
    }
    for item in re.split(r"[,;|/\n•]+", text):
        candidate = normalize_skill(item)
        if (
            2 <= len(candidate) <= 60
            and re.search(r"[\wÀ-ỹ]", candidate)
            and len(candidate.split()) <= 7
            and candidate not in {"and", "with", "year", "years", "kỹ năng"}
        ):
            found.add(candidate)
    return found


In [ ]:
def _parse_experience_numbers(text: str) -> tuple[float, float]:
    numbers = [
        float(number)
        for number in re.findall(r"\d+(?:[.,]\d+)?", text.replace(",", "."))
    ]
    if "trên" in text or "hơn" in text or ">" in text:
        lower = numbers[0] if numbers else 10.0
        return lower, float("inf")
    if len(numbers) >= 2:
        return min(numbers[0], numbers[1]), max(numbers[0], numbers[1])
    if len(numbers) == 1:
        return numbers[0], numbers[0]
    return 0.0, float("inf")


In [ ]:
def parse_required_experience(value) -> tuple[float, float] | None:
    text = normalize_text(value)
    if not text:
        return None
    if "không yêu cầu" in text:
        return 0.0, float("inf")
    return _parse_experience_numbers(text)


In [ ]:
def parse_candidate_experience(value) -> tuple[float, float] | None:
    text = normalize_text(value)
    if not text:
        return None
    if "chưa có" in text:
        return 0.0, 0.0
    return _parse_experience_numbers(text)


In [ ]:
def interval_distance(left: tuple[float, float], right: tuple[float, float]) -> float:
    left_low, left_high = left
    right_low, right_high = right
    if left_high >= right_low and right_high >= left_low:
        return 0.0
    if left_high < right_low:
        return right_low - left_high
    return left_low - right_high


In [ ]:
def experience_score(job_value, candidate_value) -> float:
    required = parse_required_experience(job_value)
    candidate = parse_candidate_experience(candidate_value)
    if required is None or candidate is None:
        return np.nan
    distance = interval_distance(required, candidate)
    return float(np.exp(-distance / max(1.0, required[0])))


In [ ]:
def location_match(candidate_value, job_value) -> float:
    candidate = normalize_location(candidate_value)
    job = normalize_location(job_value)
    if not candidate or not job:
        return 0.0
    cities = [
        "hà nội", "hồ chí minh", "đà nẵng", "cần thơ", "hải phòng",
        "bình dương", "đồng nai", "bắc giang",
    ]
    return float(any(city in candidate and city in job for city in cities) or candidate in job or job in candidate)


In [ ]:
@dataclass
class FeatureState:
    role_vectorizer: TfidfVectorizer
    description_vectorizer: TfidfVectorizer
    fit_job_ids: set[str]
    fit_candidate_ids: set[str]


In [ ]:
class ThreeSignalFeaturePipeline:
    def __init__(
        self,
        semantic_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        semantic_batch_size: int = 64,
        semantic_device: str | None = None,
        semantic_max_features: int = 6000,
        role_max_features: int = 2000,
        semantic_encoder=None,
    ):
        self.semantic_model_name = semantic_model_name
        self.semantic_batch_size = int(semantic_batch_size)
        self.semantic_device = semantic_device
        self.semantic_encoder = semantic_encoder
        self.role_vectorizer = TfidfVectorizer(max_features=role_max_features, ngram_range=(1, 2))
        self.description_vectorizer = TfidfVectorizer(
            max_features=semantic_max_features, ngram_range=(1, 2)
        )
        self.fit_job_ids: set[str] = set()
        self.fit_candidate_ids: set[str] = set()
        self.embedding_dimension: int | None = None
        self.is_fitted = False

    def _get_semantic_encoder(self):
        if self.semantic_encoder is None:
            try:
                from sentence_transformers import SentenceTransformer
            except ImportError as error:
                raise ImportError(
                    "Install sentence-transformers to compute the multilingual semantic signal"
                ) from error
            self.semantic_encoder = SentenceTransformer(
                self.semantic_model_name,
                device=self.semantic_device,
            )
        return self.semantic_encoder

    def _encode_semantic(self, documents: list[str]) -> np.ndarray:
        if not documents:
            return np.empty((0, 0), dtype=np.float32)
        embeddings = np.asarray(
            self._get_semantic_encoder().encode(
                documents,
                batch_size=self.semantic_batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_numpy=True,
            ),
            dtype=np.float32,
        )
        if embeddings.ndim != 2 or len(embeddings) != len(documents):
            raise ValueError("Semantic encoder returned an invalid embedding matrix")
        if not np.isfinite(embeddings).all():
            raise ValueError("Semantic encoder returned non-finite values")
        self.embedding_dimension = int(embeddings.shape[1])
        return embeddings

    @staticmethod
    def _job_semantic(row: pd.Series) -> str:
        return normalize_text(" ".join([
            str(row.get("Job Title", "") or ""),
            str(row.get("Job Description", "") or ""),
            str(row.get("Job Requirements", "") or ""),
        ]))

    @staticmethod
    def _candidate_semantic(row: pd.Series) -> str:
        return normalize_text(" ".join([
            str(row.get("Desired Job", "") or ""),
            str(row.get("Target", "") or ""),
            str(row.get("Skills", "") or ""),
        ]))

    @staticmethod
    def _job_description(row: pd.Series) -> str:
        return normalize_text(row.get("Job Description", ""))

    @staticmethod
    def _candidate_description(row: pd.Series) -> str:
        return normalize_text(row.get("Target", ""))

    def fit(self, train_pairs: pd.DataFrame, jobs: pd.DataFrame, candidates: pd.DataFrame):
        job_indices = set(train_pairs["job_source_index"].astype(int))
        candidate_indices = set(train_pairs["candidate_source_index"].astype(int))
        train_jobs = jobs[jobs["_source_index"].isin(job_indices)]
        train_candidates = candidates[candidates["_source_index"].isin(candidate_indices)]
        self.fit_job_ids = {f"JOB_{int(value)}" for value in train_jobs["_source_index"]}
        self.fit_candidate_ids = {f"CV_{int(value):03d}" for value in train_candidates["_source_index"]}
        role_documents = train_jobs["Job Title"].map(normalize_text).tolist()
        role_documents += train_candidates["Desired Job"].map(normalize_text).tolist()
        description_documents = [
            self._job_description(row) for _, row in train_jobs.iterrows()
        ]
        description_documents += [
            self._candidate_description(row) for _, row in train_candidates.iterrows()
        ]
        self.role_vectorizer.fit(role_documents)
        self.description_vectorizer.fit(description_documents)
        encoder = self._get_semantic_encoder()
        dimension = getattr(encoder, "get_sentence_embedding_dimension", lambda: None)()
        if dimension is None:
            probe = self._encode_semantic(["dimension probe"])
            dimension = probe.shape[1]
        self.embedding_dimension = int(dimension)
        self.is_fitted = True
        return self

    def _feature_row(self, job: pd.Series, candidate: pd.Series) -> dict:
        job_text = self._job_semantic(job)
        candidate_text = self._candidate_semantic(candidate)
        semantic_available = bool(job_text and candidate_text)
        if semantic_available:
            embeddings = self._encode_semantic([job_text, candidate_text])
            s_sem = float(np.dot(embeddings[0], embeddings[1]))
        else:
            s_sem = np.nan

        job_skills = extract_skill_set(job.get("Job Requirements", ""))
        candidate_skills = extract_skill_set(candidate.get("Skills", ""))
        skill_available = bool(job_skills) and bool(candidate_skills)
        union = job_skills | candidate_skills
        s_skill = (
            float(len(job_skills & candidate_skills) / len(union))
            if skill_available else np.nan
        )

        required_experience = parse_required_experience(
            job.get("Years of Experience", "")
        )
        candidate_experience = parse_candidate_experience(
            candidate.get("Work Experience", "")
        )
        experience_available = (
            required_experience is not None and candidate_experience is not None
        )
        s_exp = (
            experience_score(
                job.get("Years of Experience", ""),
                candidate.get("Work Experience", ""),
            )
            if experience_available else np.nan
        )

        role_score = float(cosine_similarity(
            self.role_vectorizer.transform([normalize_text(job.get("Job Title", ""))]),
            self.role_vectorizer.transform([normalize_text(candidate.get("Desired Job", ""))]),
        )[0, 0])
        description_score = float(cosine_similarity(
            self.description_vectorizer.transform([self._job_description(job)]),
            self.description_vectorizer.transform([self._candidate_description(candidate)]),
        )[0, 0])
        loc_score = location_match(candidate.get("Workplace Desired", ""), job.get("Job Address", ""))
        baseline_sem = 0.0 if not np.isfinite(s_sem) else s_sem
        baseline_skill = 0.0 if not np.isfinite(s_skill) else s_skill
        baseline_exp = 0.0 if not np.isfinite(s_exp) else s_exp
        heuristic_score = (
            0.30 * loc_score + 0.25 * baseline_skill + 0.20 * baseline_exp
            + 0.15 * role_score + 0.10 * description_score
        )
        return {
            "s_sem": s_sem,
            "s_skill": s_skill,
            "s_exp": s_exp,
            "sem_available": semantic_available,
            "skill_available": skill_available,
            "exp_available": experience_available,
            "baseline_location": loc_score,
            "baseline_skill": baseline_skill,
            "baseline_experience": baseline_exp,
            "baseline_role": role_score,
            "baseline_description": description_score,
            "heuristic_score": float(heuristic_score),
        }

    def transform(self, pairs: pd.DataFrame, jobs: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
        """Transform pairs with one vectorizer call per entity type, not per pair."""
        if not self.is_fitted:
            raise RuntimeError("Feature pipeline must be fitted on train")
        if pairs.empty:
            return pairs.copy()

        job_indices = pd.Index(pd.unique(pairs["job_source_index"].astype(int)))
        candidate_indices = pd.Index(pd.unique(pairs["candidate_source_index"].astype(int)))
        job_entities = jobs.set_index("_source_index").loc[job_indices]
        candidate_entities = candidates.set_index("_source_index").loc[candidate_indices]

        job_semantic_text = [self._job_semantic(row) for _, row in job_entities.iterrows()]
        candidate_semantic_text = [self._candidate_semantic(row) for _, row in candidate_entities.iterrows()]
        job_semantic_matrix = self._encode_semantic(job_semantic_text)
        candidate_semantic_matrix = self._encode_semantic(candidate_semantic_text)

        job_role_text = job_entities["Job Title"].map(normalize_text).tolist()
        candidate_role_text = candidate_entities["Desired Job"].map(normalize_text).tolist()
        job_role_matrix = self.role_vectorizer.transform(job_role_text)
        candidate_role_matrix = self.role_vectorizer.transform(candidate_role_text)

        job_description_text = [
            self._job_description(row) for _, row in job_entities.iterrows()
        ]
        candidate_description_text = [
            self._candidate_description(row) for _, row in candidate_entities.iterrows()
        ]
        job_description_matrix = self.description_vectorizer.transform(
            job_description_text
        )
        candidate_description_matrix = self.description_vectorizer.transform(
            candidate_description_text
        )

        job_positions = pairs["job_source_index"].astype(int).map(
            {value: position for position, value in enumerate(job_indices)}
        ).to_numpy(int)
        candidate_positions = pairs["candidate_source_index"].astype(int).map(
            {value: position for position, value in enumerate(candidate_indices)}
        ).to_numpy(int)

        semantic_available_by_job = np.asarray([bool(text) for text in job_semantic_text])
        semantic_available_by_candidate = np.asarray([bool(text) for text in candidate_semantic_text])
        semantic_available = (
            semantic_available_by_job[job_positions]
            & semantic_available_by_candidate[candidate_positions]
        )
        semantic_scores = np.einsum(
            "ij,ij->i",
            job_semantic_matrix[job_positions],
            candidate_semantic_matrix[candidate_positions],
        )
        semantic_scores[~semantic_available] = np.nan

        role_scores = np.asarray(
            job_role_matrix[job_positions]
            .multiply(candidate_role_matrix[candidate_positions])
            .sum(axis=1)
        ).ravel()
        description_scores = np.asarray(
            job_description_matrix[job_positions]
            .multiply(candidate_description_matrix[candidate_positions])
            .sum(axis=1)
        ).ravel()

        job_skills = [extract_skill_set(value) for value in job_entities["Job Requirements"]]
        candidate_skills = [extract_skill_set(value) for value in candidate_entities["Skills"]]
        job_experience = [normalize_text(value) for value in job_entities["Years of Experience"]]
        candidate_experience = [normalize_text(value) for value in candidate_entities["Work Experience"]]
        job_locations = [value for value in job_entities["Job Address"]]
        candidate_locations = [value for value in candidate_entities["Workplace Desired"]]

        skill_scores = np.empty(len(pairs), dtype=float)
        experience_scores = np.empty(len(pairs), dtype=float)
        location_scores = np.empty(len(pairs), dtype=float)
        skill_available = np.empty(len(pairs), dtype=bool)
        experience_available = np.empty(len(pairs), dtype=bool)
        for row_index, (job_position, candidate_position) in enumerate(
            zip(job_positions, candidate_positions)
        ):
            left_skills = job_skills[job_position]
            right_skills = candidate_skills[candidate_position]
            union = left_skills | right_skills
            skill_available[row_index] = bool(left_skills) and bool(right_skills)
            skill_scores[row_index] = (
                len(left_skills & right_skills) / len(union)
                if skill_available[row_index] else np.nan
            )
            job_exp = job_experience[job_position]
            candidate_exp = candidate_experience[candidate_position]
            experience_available[row_index] = (
                parse_required_experience(job_exp) is not None
                and parse_candidate_experience(candidate_exp) is not None
            )
            experience_scores[row_index] = (
                experience_score(job_exp, candidate_exp)
                if experience_available[row_index] else np.nan
            )
            location_scores[row_index] = location_match(
                candidate_locations[candidate_position], job_locations[job_position]
            )

        output = pairs.copy().reset_index(drop=True)
        output["s_sem"] = semantic_scores
        output["s_skill"] = skill_scores
        output["s_exp"] = experience_scores
        output["sem_available"] = semantic_available
        output["skill_available"] = skill_available
        output["exp_available"] = experience_available
        output["baseline_location"] = location_scores
        output["baseline_skill"] = np.nan_to_num(skill_scores, nan=0.0)
        output["baseline_experience"] = np.nan_to_num(experience_scores, nan=0.0)
        output["baseline_role"] = role_scores
        output["baseline_description"] = description_scores
        output["heuristic_score"] = (
            0.30 * output["baseline_location"]
            + 0.25 * output["baseline_skill"]
            + 0.20 * output["baseline_experience"]
            + 0.15 * output["baseline_role"]
            + 0.10 * output["baseline_description"]
        )
        return output

    def transform_gold(self, gold: pd.DataFrame, raw: RawData) -> pd.DataFrame:
        pairs = gold_pairs_to_raw_indices(gold)
        features = self.transform(pairs, raw.jobs, raw.candidates)
        metadata = gold.drop(columns=[column for column in features.columns if column in gold.columns and column not in {"pair_id", "job_id", "cand_id"}])
        merged = metadata.merge(features, on=["pair_id", "job_id", "cand_id"], how="inner")
        if len(merged) != len(gold):
            raise ValueError("Gold feature transform lost rows")
        return merged

    def impute_for_models(self, frame: pd.DataFrame, fill_values: dict[str, float] | None = None) -> tuple[pd.DataFrame, dict[str, float]]:
        output = frame.copy()
        if fill_values is None:
            fill_values = {
                column: float(output[column].median()) if output[column].notna().any() else 0.0
                for column in FEATURE_COLUMNS
            }
        output[FEATURE_COLUMNS] = output[FEATURE_COLUMNS].fillna(fill_values)
        if not np.isfinite(output[FEATURE_COLUMNS].to_numpy(float)).all():
            raise ValueError("Non-finite model feature")
        return output, dict(fill_values)

    @staticmethod
    def _sentence_transformers_version() -> str | None:
        try:
            return importlib.metadata.version("sentence-transformers")
        except importlib.metadata.PackageNotFoundError:
            return None

    def manifest(self) -> dict:
        if not self.is_fitted:
            raise RuntimeError("Feature pipeline is not fitted")
        role_terms = "\n".join(sorted(self.role_vectorizer.vocabulary_))
        description_terms = "\n".join(
            sorted(self.description_vectorizer.vocabulary_)
        )
        return {
            "feature_columns": list(FEATURE_COLUMNS),
            "baseline_components": [
                "baseline_location", "baseline_skill", "baseline_experience",
                "baseline_role", "baseline_description",
            ],
            "signal_definition_version": "multilingual-sentence-embedding-v2",
            "semantic_encoder": {
                "model_name": self.semantic_model_name,
                "batch_size": self.semantic_batch_size,
                "device": self.semantic_device,
                "embedding_dimension": self.embedding_dimension,
                "sentence_transformers_version": self._sentence_transformers_version(),
            },
            "signal_field_mapping": {
                "s_sem": [
                    "Job Title + Job Description + Job Requirements",
                    "Desired Job + Target + Skills",
                ],
                "s_skill": ["Job Requirements", "Skills"],
                "s_exp": ["Years of Experience", "Work Experience"],
            },
            "baseline_definition_version": "independent-five-component-v2",
            "baseline_field_mapping": {
                "location": ["Job Address", "Workplace Desired"],
                "skill": ["Job Requirements", "Skills"],
                "experience": ["Years of Experience", "Work Experience"],
                "role": ["Job Title", "Desired Job"],
                "description": ["Job Description", "Target"],
            },
            "fit_job_ids": sorted(self.fit_job_ids),
            "fit_candidate_ids": sorted(self.fit_candidate_ids),
            "role_lexical_vocabulary_size": len(self.role_vectorizer.vocabulary_),
            "description_lexical_vocabulary_size": len(
                self.description_vectorizer.vocabulary_
            ),
            "role_lexical_vocabulary_sha256": hashlib.sha256(role_terms.encode()).hexdigest(),
            "description_lexical_vocabulary_sha256": hashlib.sha256(
                description_terms.encode()
            ).hexdigest(),
        }


## 5. Labeling functions và mô hình nhãn Dawid–Skene

Mỗi tín hiệu tạo một LF ba trạng thái `{−1, 0, +1}`. Ngưỡng 25/75 percentile chỉ fit trên development-train; `0` là abstain. EM ước lượng prior, sensitivity và specificity trong khi loại abstention khỏi mẫu số tương ứng. Các tham số bị đóng băng khi suy luận held-out.


In [ ]:
_SIGNAL_TO_LF = {"s_sem": "lf_sem", "s_skill": "lf_skill", "s_exp": "lf_exp"}


In [ ]:
_AVAILABILITY = {"s_sem": "sem_available", "s_skill": "skill_available", "s_exp": "exp_available"}


In [ ]:
class PercentileLabelingFunctions:
    POLICY = "negative-global-positive-tail-v1"

    def __init__(
        self,
        negative_percentile: float = 25,
        positive_percentile: float = 75,
        threshold_policy: str = POLICY,
    ):
        if threshold_policy != self.POLICY:
            raise ValueError(f"Unsupported threshold policy: {threshold_policy}")
        self.negative_percentile = negative_percentile
        self.positive_percentile = positive_percentile
        self.threshold_policy = threshold_policy
        self.thresholds: dict[str, dict[str, float]] = {}
        self.threshold_diagnostics: dict[str, dict[str, int]] = {}
        self.is_fitted = False

    def fit(self, train: pd.DataFrame):
        thresholds = {}
        diagnostics = {}
        for signal in _SIGNAL_TO_LF:
            active = train.loc[train[_AVAILABILITY[signal]].astype(bool), signal].dropna().to_numpy(float)
            if not len(active):
                raise ValueError(f"No active train observations for {signal}")
            negative = float(np.percentile(active, self.negative_percentile))
            positive_tail = active[active > negative]
            if not len(positive_tail):
                raise ValueError(f"{signal} has no variation for labeling functions")
            positive = float(np.percentile(positive_tail, self.positive_percentile))
            thresholds[signal] = {"negative": negative, "positive": positive}
            diagnostics[signal] = {
                "n_active": int(len(active)),
                "n_at_or_below_negative": int((active <= negative).sum()),
                "n_positive_tail": int(len(positive_tail)),
            }
        self.thresholds = thresholds
        self.threshold_diagnostics = diagnostics
        self.is_fitted = True
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if not self.is_fitted:
            raise RuntimeError("Labeling functions must be fitted on train")
        output = frame.copy()
        for signal, lf_column in _SIGNAL_TO_LF.items():
            available = output[_AVAILABILITY[signal]].astype(bool) & output[signal].notna()
            values = output[signal].to_numpy(float)
            negative = self.thresholds[signal]["negative"]
            positive = self.thresholds[signal]["positive"]
            votes = np.zeros(len(output), dtype=int)
            votes[available.to_numpy() & (values <= negative)] = -1
            votes[available.to_numpy() & (values >= positive)] = 1
            output[lf_column] = votes
        return output


In [ ]:
class DawidSkeneThreeSource:
    def __init__(
        self,
        n_iter: int = 100,
        parameter_clip: tuple[float, float] = (0.51, 0.99),
        prior_clip: tuple[float, float] = (0.05, 0.95),
        initial_accuracy: float = 0.75,
        convergence_tolerance: float = 1e-7,
    ):
        if int(n_iter) <= 0:
            raise ValueError("Dawid--Skene n_iter must be positive")
        if not 0.5 < parameter_clip[0] < parameter_clip[1] < 1.0:
            raise ValueError("parameter_clip must satisfy 0.5 < lower < upper < 1")
        if not 0.0 < prior_clip[0] < prior_clip[1] < 1.0:
            raise ValueError("prior_clip must satisfy 0 < lower < upper < 1")
        if not 0.5 < initial_accuracy < 1.0:
            raise ValueError("initial_accuracy must be in (0.5, 1)")
        if convergence_tolerance <= 0:
            raise ValueError("convergence_tolerance must be positive")
        self.n_iter = int(n_iter)
        self.parameter_clip = tuple(float(value) for value in parameter_clip)
        self.prior_clip = tuple(float(value) for value in prior_clip)
        self.initial_accuracy = float(initial_accuracy)
        self.convergence_tolerance = float(convergence_tolerance)
        self.prior = 0.5
        self.sensitivities: np.ndarray | None = None
        self.specificities: np.ndarray | None = None
        self.is_fitted = False

    @staticmethod
    def _posterior(matrix: np.ndarray, prior: float, sensitivity: np.ndarray, specificity: np.ndarray) -> np.ndarray:
        positive = matrix == 1
        negative = matrix == -1
        log_y1 = np.full(len(matrix), np.log(prior + 1e-12), dtype=float)
        log_y0 = np.full(len(matrix), np.log(1.0 - prior + 1e-12), dtype=float)
        for index in range(matrix.shape[1]):
            log_y1 += positive[:, index] * np.log(sensitivity[index] + 1e-12)
            log_y1 += negative[:, index] * np.log(1.0 - sensitivity[index] + 1e-12)
            log_y0 += positive[:, index] * np.log(1.0 - specificity[index] + 1e-12)
            log_y0 += negative[:, index] * np.log(specificity[index] + 1e-12)
        maximum = np.maximum(log_y1, log_y0)
        y1 = np.exp(log_y1 - maximum)
        y0 = np.exp(log_y0 - maximum)
        return y1 / (y1 + y0 + 1e-12)

    def fit(self, lf_frame: pd.DataFrame):
        matrix = lf_frame[LF_COLUMNS].to_numpy(int)
        if not len(matrix):
            raise ValueError("Cannot fit Dawid--Skene on empty data")
        active = matrix != 0
        active_rows = active.any(axis=1)
        if active_rows.any():
            positives = (matrix[active_rows] == 1).sum(axis=1)
            negatives = (matrix[active_rows] == -1).sum(axis=1)
            vote_direction = np.where(
                positives > negatives,
                1.0,
                np.where(negatives > positives, 0.0, np.nan),
            )
            prior = (
                float(np.nanmean(vote_direction))
                if np.isfinite(vote_direction).any()
                else 0.5
            )
        else:
            prior = 0.5
        prior = float(np.clip(prior, *self.prior_clip))
        sensitivity = np.full(
            matrix.shape[1], self.initial_accuracy, dtype=float
        )
        specificity = np.full(
            matrix.shape[1], self.initial_accuracy, dtype=float
        )
        previous = None
        for _ in range(self.n_iter):
            posterior = self._posterior(matrix, prior, sensitivity, specificity)
            prior = float(np.clip(posterior.mean(), 0.05, 0.95))
            for index in range(matrix.shape[1]):
                mask = active[:, index]
                positive_denom = posterior[mask].sum()
                negative_denom = (1.0 - posterior[mask]).sum()
                if positive_denom > 1e-12:
                    sensitivity[index] = np.clip(
                        (posterior * (matrix[:, index] == 1)).sum() / positive_denom,
                        *self.parameter_clip,
                    )
                if negative_denom > 1e-12:
                    specificity[index] = np.clip(
                        ((1.0 - posterior) * (matrix[:, index] == -1)).sum() / negative_denom,
                        *self.parameter_clip,
                    )
            if (
                previous is not None
                and np.max(np.abs(posterior - previous))
                < self.convergence_tolerance
            ):
                break
            previous = posterior
        self.prior = prior
        self.sensitivities = sensitivity
        self.specificities = specificity
        self.is_fitted = True
        return self

    def predict_proba(self, lf_frame: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Dawid--Skene must be fitted on train")
        return self._posterior(
            lf_frame[LF_COLUMNS].to_numpy(int),
            self.prior,
            self.sensitivities,
            self.specificities,
        )

    def parameters(self) -> dict:
        if not self.is_fitted:
            raise RuntimeError("Dawid--Skene is not fitted")
        return {
            "prior": float(self.prior),
            "lf_columns": list(LF_COLUMNS),
            "sensitivities": self.sensitivities.tolist(),
            "specificities": self.specificities.tolist(),
            "assumptions": {
                "n_iter": self.n_iter,
                "parameter_clip": list(self.parameter_clip),
                "prior_clip": list(self.prior_clip),
                "initial_accuracy": self.initial_accuracy,
                "convergence_tolerance": self.convergence_tolerance,
            },
        }


In [ ]:
class ThreeSourceWeakLabelPipeline:
    def __init__(
        self,
        negative_percentile: float = 25,
        positive_percentile: float = 75,
        threshold_policy: str = PercentileLabelingFunctions.POLICY,
        label_model_config: dict | None = None,
    ):
        self.labeling_functions = PercentileLabelingFunctions(
            negative_percentile, positive_percentile, threshold_policy
        )
        self.label_model = DawidSkeneThreeSource(**(label_model_config or {}))
        self.is_fitted = False

    def fit_transform_train(self, train: pd.DataFrame) -> pd.DataFrame:
        lfs = self.labeling_functions.fit(train).transform(train)
        self.label_model.fit(lfs)
        self.is_fitted = True
        output = lfs.copy()
        output["y_prob"] = self.label_model.predict_proba(lfs)
        return output

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if not self.is_fitted:
            raise RuntimeError("Weak-label pipeline must be fitted on train")
        lfs = self.labeling_functions.transform(frame)
        output = lfs.copy()
        output["y_prob"] = self.label_model.predict_proba(lfs)
        return output

    def parameters(self) -> dict:
        return {
            "threshold_policy": self.labeling_functions.threshold_policy,
            "negative_percentile": self.labeling_functions.negative_percentile,
            "positive_percentile": self.labeling_functions.positive_percentile,
            "thresholds": self.labeling_functions.thresholds,
            "threshold_diagnostics": self.labeling_functions.threshold_diagnostics,
            "label_model": self.label_model.parameters(),
        }


In [ ]:
def strict_three_of_three(lf_frame: pd.DataFrame) -> np.ndarray:
    matrix = lf_frame[LF_COLUMNS].to_numpy(int)
    prediction = np.full(len(matrix), np.nan)
    prediction[(matrix == 1).all(axis=1)] = 1.0
    prediction[(matrix == -1).all(axis=1)] = 0.0
    return prediction


In [ ]:
def lf_diagnostics(lf_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    stats = []
    for column in LF_COLUMNS:
        values = lf_frame[column].to_numpy(int)
        stats.append({
            "lf": column,
            "n_total": int(len(values)),
            "n_positive": int((values == 1).sum()),
            "n_negative": int((values == -1).sum()),
            "n_abstain": int((values == 0).sum()),
            "coverage": float(np.mean(values != 0)),
            "positive_rate": float(np.mean(values == 1)),
            "negative_rate": float(np.mean(values == -1)),
            "abstain_rate": float(np.mean(values == 0)),
        })
    pair_rows = []
    for left, right in itertools.combinations(LF_COLUMNS, 2):
        left_values = lf_frame[left].to_numpy(int)
        right_values = lf_frame[right].to_numpy(int)
        joint = (left_values != 0) & (right_values != 0)
        if joint.any():
            joint_left = left_values[joint]
            joint_right = right_values[joint]
            if len(np.unique(joint_left)) < 2 or len(np.unique(joint_right)) < 2:
                spearman = 0.0
            else:
                rho, _ = spearmanr(joint_left, joint_right)
                spearman = float(np.nan_to_num(rho))
            agreement = float(np.mean(joint_left == joint_right))
            conflict = float(np.mean(joint_left != joint_right))
        else:
            agreement = np.nan
            conflict = np.nan
            spearman = np.nan
        pair_rows.append({
            "lf_a": left,
            "lf_b": right,
            "joint_coverage": float(joint.mean()),
            "agreement": agreement,
            "conflict": conflict,
            "spearman": spearman,
        })
    return pd.DataFrame(stats), pd.DataFrame(pair_rows)


In [ ]:
def binary_quality(y_true: np.ndarray, predictions: np.ndarray) -> dict:
    observed = np.isfinite(predictions)
    truth_all = np.asarray(y_true, dtype=int)
    if not observed.any():
        return {
            "precision": np.nan,
            "recall": np.nan,
            "coverage": 0.0,
            "n_covered": 0,
            "n_positive": int(truth_all.sum()),
            "n_predicted_positive": 0,
            "tp": 0,
            "fp": 0,
            "fn": int(truth_all.sum()),
            "tn": int((truth_all == 0).sum()),
        }
    truth = truth_all[observed]
    predicted = np.asarray(predictions, dtype=float)[observed].astype(int)
    tp = int(((truth == 1) & (predicted == 1)).sum())
    fp = int(((truth == 0) & (predicted == 1)).sum())
    fn = int(((truth == 1) & (predicted == 0)).sum())
    tn = int(((truth == 0) & (predicted == 0)).sum())
    return {
        "precision": float(precision_score(truth, predicted, zero_division=0)),
        "recall": float(recall_score(truth, predicted, zero_division=0)),
        "coverage": float(observed.mean()),
        "n_covered": int(observed.sum()),
        "n_positive": int(truth.sum()),
        "n_predicted_positive": int(predicted.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


## 6. Pointwise, pairwise và listwise LTR

Ba formulation dùng cùng một linear scorer ba chiều:

- **Pointwise:** soft binary cross-entropy với posterior Dawid–Skene.
- **Pairwise:** RankNet trên cặp cùng job, với chênh lệch posterior cố định tối thiểu 0,02.
- **Listwise:** ListNet top-one cross-entropy theo query, với target là softmax của logit posterior đã clip.

Batch unit được khai báo riêng: CV–job pairs cho pointwise, preference pairs cho pairwise và whole queries cho listwise. Hyperparameter và early stopping tối đa hóa macro weak-validation nDCG@5. Gold-validation chọn formulation theo nDCG@5; các formulation nằm trong 0,005 của điểm cao nhất được so bằng common validation RankNet loss, rồi mới ưu tiên mô hình đơn giản hơn.


In [ ]:
class LinearScorer(nn.Module):
    """Shared scorer capacity for pointwise, pairwise, and listwise objectives."""

    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(len(FEATURE_COLUMNS), 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.linear(features).squeeze(-1)


In [ ]:
@dataclass
class PreparedDataset:
    features: torch.Tensor
    targets: torch.Tensor
    query_order: tuple[object, ...]
    query_indices: dict[object, torch.Tensor]


In [ ]:
@dataclass
class PairwiseState:
    train_table: pd.DataFrame
    validation_table: pd.DataFrame
    train_tensors: tuple[torch.Tensor, torch.Tensor]
    validation_tensors: tuple[torch.Tensor, torch.Tensor]
    train_hash: str
    validation_hash: str


In [ ]:
@dataclass
class TrainingResult:
    formulation: str
    model: LinearScorer
    learning_rate: float
    batch_size: int
    best_epoch: int
    best_validation_loss: float
    best_train_loss: float
    epochs_ran: int
    stopped_early: bool
    weight_decay: float
    gradient_clip_norm: float
    selection_metric: str
    best_validation_ndcg_at_5: float
    batch_unit: str
    device: str
    history: pd.DataFrame
    train_pair_hash: str | None = None
    validation_pair_hash: str | None = None

    def metadata(self) -> dict:
        return {
            "formulation": self.formulation,
            "learning_rate": self.learning_rate,
            "batch_size": self.batch_size,
            "best_epoch": self.best_epoch,
            "best_validation_loss": self.best_validation_loss,
            "best_train_loss": self.best_train_loss,
            "generalization_gap": self.best_validation_loss - self.best_train_loss,
            "epochs_ran": self.epochs_ran,
            "stopped_early": self.stopped_early,
            "weight_decay": self.weight_decay,
            "gradient_clip_norm": self.gradient_clip_norm,
            "selection_metric": self.selection_metric,
            "best_validation_ndcg_at_5": self.best_validation_ndcg_at_5,
            "batch_unit": self.batch_unit,
            "device": self.device,
            "train_pair_hash": self.train_pair_hash,
            "validation_pair_hash": self.validation_pair_hash,
        }


In [ ]:
def _resolve_device(device: str | torch.device = "cpu") -> torch.device:
    requested = torch.device(device)
    if requested.type == "cpu":
        return requested
    if requested.type != "cuda":
        raise ValueError(f"Unsupported training device: {requested}")
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA was requested but is not available in this PyTorch environment"
        )
    index = 0 if requested.index is None else requested.index
    if index < 0 or index >= torch.cuda.device_count():
        raise ValueError(
            f"CUDA device index {index} is outside the available device range"
        )
    return torch.device(f"cuda:{index}")


In [ ]:
def _model_device(model: LinearScorer) -> torch.device:
    return next(model.parameters()).device


In [ ]:
def _tensor_features(frame: pd.DataFrame) -> torch.Tensor:
    return torch.tensor(frame[FEATURE_COLUMNS].to_numpy(float), dtype=torch.float32)


In [ ]:
def _materialize_dataset(
    frame: pd.DataFrame,
    device: str | torch.device = "cpu",
) -> PreparedDataset:
    resolved = _resolve_device(device)
    features = _tensor_features(frame).to(resolved)
    targets = torch.tensor(
        frame["y_prob"].to_numpy(float), dtype=torch.float32, device=resolved
    )
    job_ids = frame["job_id"].to_numpy()
    query_order = tuple(sorted(pd.unique(job_ids).tolist()))
    query_indices = {
        job_id: torch.tensor(
            np.flatnonzero(job_ids == job_id), dtype=torch.long, device=resolved
        )
        for job_id in query_order
    }
    return PreparedDataset(features, targets, query_order, query_indices)


In [ ]:
def predict_scores(
    model: LinearScorer,
    data: pd.DataFrame | PreparedDataset,
) -> np.ndarray:
    model.eval()
    device = _model_device(model)
    features = (
        data.features.to(device)
        if isinstance(data, PreparedDataset)
        else _tensor_features(data).to(device)
    )
    with torch.no_grad():
        return model(features).detach().cpu().numpy()


In [ ]:
def _batches(size: int, batch_size: int, rng: np.random.RandomState):
    indices = np.arange(size)
    rng.shuffle(indices)
    for start in range(0, size, batch_size):
        yield indices[start:start + batch_size]


In [ ]:
def build_pair_table(
    frame: pd.DataFrame,
    pair_delta: float,
    max_pairs_per_job: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    rows = []
    for job_id, group in frame.groupby("job_id", sort=True):
        group = group.sort_values(["cand_id", "pair_id"], kind="mergesort")
        indices = group.index.to_numpy(int)
        probabilities = group["y_prob"].to_numpy(float)
        pair_ids = group["pair_id"].to_numpy(int)
        left_positions, right_positions = np.triu_indices(len(group), k=1)
        differences = probabilities[left_positions] - probabilities[right_positions]
        eligible_mask = (~np.isclose(differences, 0.0)) & (np.abs(differences) >= pair_delta)
        left_positions = left_positions[eligible_mask]
        right_positions = right_positions[eligible_mask]
        differences = differences[eligible_mask]
        if len(differences) > max_pairs_per_job:
            selected = rng.choice(
                len(differences), size=max_pairs_per_job, replace=False
            )
            left_positions = left_positions[selected]
            right_positions = right_positions[selected]
            differences = differences[selected]
        positive = differences > 0
        preferred_positions = np.where(positive, left_positions, right_positions)
        nonpreferred_positions = np.where(positive, right_positions, left_positions)
        eligible = [
            {
                "job_id": job_id,
                "preferred_index": int(indices[preferred]),
                "nonpreferred_index": int(indices[nonpreferred]),
                "preferred_pair_id": int(pair_ids[preferred]),
                "nonpreferred_pair_id": int(pair_ids[nonpreferred]),
                "delta": float(abs(difference)),
            }
            for preferred, nonpreferred, difference in zip(
                preferred_positions, nonpreferred_positions, differences
            )
        ]
        rows.extend(sorted(
            eligible,
            key=lambda row: (
                row["job_id"], row["preferred_pair_id"],
                row["nonpreferred_pair_id"],
            ),
        ))
    table = pd.DataFrame(rows)
    if table.empty:
        raise ValueError(f"No RankNet pairs at fixed delta {pair_delta}; no fallback is permitted")
    if (table["delta"] < pair_delta).any():
        raise AssertionError("Pair below fixed delta")
    return table.reset_index(drop=True)


In [ ]:
def pair_table_hash(table: pd.DataFrame) -> str:
    columns = ["job_id", "preferred_pair_id", "nonpreferred_pair_id"]
    payload = table[columns].sort_values(columns, kind="mergesort").to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


In [ ]:
def _pair_tensors(
    frame: pd.DataFrame,
    table: pd.DataFrame,
    device: str | torch.device = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    resolved = _resolve_device(device)
    preferred = frame.loc[table["preferred_index"].astype(int), FEATURE_COLUMNS].to_numpy(float)
    nonpreferred = frame.loc[table["nonpreferred_index"].astype(int), FEATURE_COLUMNS].to_numpy(float)
    return (
        torch.tensor(preferred, dtype=torch.float32, device=resolved),
        torch.tensor(nonpreferred, dtype=torch.float32, device=resolved),
    )


In [ ]:
def pointwise_loss(model: LinearScorer, frame: pd.DataFrame) -> torch.Tensor:
    device = _model_device(model)
    target = torch.tensor(
        frame["y_prob"].to_numpy(float), dtype=torch.float32, device=device
    )
    return F.binary_cross_entropy_with_logits(
        model(_tensor_features(frame).to(device)), target
    )


In [ ]:
def _pointwise_tensor_loss(
    model: LinearScorer,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(model(features), targets)


In [ ]:
def pairwise_loss(model: LinearScorer, preferred: torch.Tensor, nonpreferred: torch.Tensor) -> torch.Tensor:
    return F.softplus(-(model(preferred) - model(nonpreferred))).mean()


In [ ]:
def listnet_target_distribution(
    probabilities: torch.Tensor,
    temperature: float = 1.0,
    epsilon: float = 1e-4,
) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError("Listwise temperature must be positive")
    if not 0 < epsilon < 0.5:
        raise ValueError("Listwise logit epsilon must be in (0, 0.5)")
    clipped = probabilities.clamp(epsilon, 1.0 - epsilon)
    logits = torch.log(clipped) - torch.log1p(-clipped)
    return torch.softmax(logits / temperature, dim=0)


In [ ]:
def listwise_loss(
    model: LinearScorer,
    frame: pd.DataFrame,
    temperature: float = 1.0,
    logit_epsilon: float = 1e-4,
) -> torch.Tensor:
    losses = []
    device = _model_device(model)
    features = _tensor_features(frame).to(device)
    positions = pd.Series(np.arange(len(frame)), index=frame.index)
    for _, group in frame.groupby("job_id", sort=True):
        index = torch.tensor(
            positions.loc[group.index].to_numpy(int),
            dtype=torch.long,
            device=device,
        )
        targets = torch.tensor(
            group["y_prob"].to_numpy(float),
            dtype=torch.float32,
            device=device,
        )
        target_distribution = listnet_target_distribution(
            targets, temperature, logit_epsilon
        )
        log_prediction = torch.log_softmax(model(features[index]), dim=0)
        losses.append(-(target_distribution * log_prediction).sum())
    if not losses:
        raise ValueError("Listwise loss requires at least one query")
    return torch.stack(losses).mean()


In [ ]:
def _prepared_listwise_loss(
    model: LinearScorer,
    prepared: PreparedDataset,
    query_ids,
    temperature: float,
    logit_epsilon: float,
) -> torch.Tensor:
    losses = []
    for query_id in query_ids:
        index = prepared.query_indices[query_id]
        target_distribution = listnet_target_distribution(
            prepared.targets[index], temperature, logit_epsilon
        )
        log_prediction = torch.log_softmax(model(prepared.features[index]), dim=0)
        losses.append(-(target_distribution * log_prediction).sum())
    if not losses:
        raise ValueError("Listwise loss requires at least one query")
    return torch.stack(losses).mean()


In [ ]:
def _optimizer_step(
    optimizer: torch.optim.Optimizer,
    model: LinearScorer,
    loss: torch.Tensor,
    gradient_clip_norm: float,
) -> None:
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
    optimizer.step()


In [ ]:
def _listwise_query_batches(
    frame: pd.DataFrame,
    queries_per_batch: int,
    query_ids: np.ndarray | None = None,
):
    if queries_per_batch <= 0:
        raise ValueError("Listwise queries per batch must be positive")
    ordered = (
        np.asarray(query_ids)
        if query_ids is not None
        else np.asarray(sorted(frame["job_id"].unique()))
    )
    for start in range(0, len(ordered), queries_per_batch):
        query_batch = set(ordered[start:start + queries_per_batch])
        yield frame[frame["job_id"].isin(query_batch)]


In [ ]:
def _query_batches(query_ids: np.ndarray, queries_per_batch: int):
    if queries_per_batch <= 0:
        raise ValueError("Listwise queries per batch must be positive")
    for start in range(0, len(query_ids), queries_per_batch):
        yield query_ids[start:start + queries_per_batch]


In [ ]:
def _run_training_epoch(
    formulation: str,
    model: LinearScorer,
    optimizer: torch.optim.Optimizer,
    train: PreparedDataset,
    train_pair_tensors: tuple[torch.Tensor, torch.Tensor] | None,
    batch_size: int,
    listwise_temperature: float,
    listwise_logit_epsilon: float,
    gradient_clip_norm: float,
    rng: np.random.RandomState,
) -> None:
    model.train()
    if formulation == "pointwise":
        for indices in _batches(len(train.features), batch_size, rng):
            loss = _pointwise_tensor_loss(
                model, train.features[indices], train.targets[indices]
            )
            _optimizer_step(optimizer, model, loss, gradient_clip_norm)
    elif formulation == "pairwise":
        preferred, nonpreferred = train_pair_tensors
        for indices in _batches(len(preferred), batch_size, rng):
            loss = pairwise_loss(model, preferred[indices], nonpreferred[indices])
            _optimizer_step(optimizer, model, loss, gradient_clip_norm)
    elif formulation == "listwise":
        query_ids = np.asarray(train.query_order, dtype=object)
        rng.shuffle(query_ids)
        for batch in _query_batches(query_ids, batch_size):
            loss = _prepared_listwise_loss(
                model, train, batch, listwise_temperature,
                listwise_logit_epsilon,
            )
            _optimizer_step(optimizer, model, loss, gradient_clip_norm)
    else:
        raise ValueError(f"Unknown formulation: {formulation}")


In [ ]:
def _objective_loss(
    formulation: str,
    model: LinearScorer,
    prepared: PreparedDataset,
    pair_tensors: tuple[torch.Tensor, torch.Tensor] | None,
    listwise_temperature: float,
    listwise_logit_epsilon: float,
) -> float:
    if formulation == "pointwise":
        loss = _pointwise_tensor_loss(model, prepared.features, prepared.targets)
    elif formulation == "pairwise":
        loss = pairwise_loss(model, *pair_tensors)
    else:
        loss = _prepared_listwise_loss(
            model, prepared, prepared.query_order,
            listwise_temperature, listwise_logit_epsilon,
        )
    return float(loss.item())


In [ ]:
def _build_pairwise_state(
    formulation: str,
    train: pd.DataFrame,
    validation: pd.DataFrame,
    pair_delta: float,
    max_pairs_per_job: int,
    seed: int,
    device: str | torch.device = "cpu",
) -> PairwiseState | None:
    if formulation != "pairwise":
        return None
    train_table = build_pair_table(train, pair_delta, max_pairs_per_job, seed)
    validation_table = build_pair_table(
        validation, pair_delta, max_pairs_per_job, seed + 1
    )
    return PairwiseState(
        train_table=train_table,
        validation_table=validation_table,
        train_tensors=_pair_tensors(train, train_table, device),
        validation_tensors=_pair_tensors(validation, validation_table, device),
        train_hash=pair_table_hash(train_table),
        validation_hash=pair_table_hash(validation_table),
    )


In [ ]:
def _weak_validation_ndcg_at_5(
    model: LinearScorer,
    prepared: PreparedDataset,
) -> float:
    scores = predict_scores(model, prepared)
    relevance = prepared.targets.cpu().numpy()
    values = []
    for query_id in prepared.query_order:
        index = prepared.query_indices[query_id].cpu().numpy()
        query_relevance = relevance[index]
        query_scores = scores[index]
        order = np.argsort(-query_scores, kind="mergesort")
        ideal_order = np.argsort(-query_relevance, kind="mergesort")
        discounts = np.log2(np.arange(2, min(5, len(index)) + 2))
        actual = np.sum(
            (np.power(2.0, query_relevance[order][:5]) - 1.0) / discounts
        )
        ideal = np.sum(
            (np.power(2.0, query_relevance[ideal_order][:5]) - 1.0) / discounts
        )
        values.append(0.0 if ideal == 0.0 else float(actual / ideal))
    if not values:
        raise ValueError("Weak-validation nDCG@5 requires at least one query")
    return float(np.mean(values))


In [ ]:
def _batch_unit(formulation: str) -> str:
    return {
        "pointwise": "cv_job_pairs",
        "pairwise": "preference_pairs",
        "listwise": "queries",
    }[formulation]


In [ ]:
def _train_one(
    formulation: str,
    train: pd.DataFrame,
    validation: pd.DataFrame,
    seed: int,
    learning_rate: float,
    batch_size: int,
    max_epochs: int,
    patience: int,
    pair_delta: float,
    max_pairs_per_job: int,
    listwise_temperature: float,
    listwise_logit_epsilon: float,
    weight_decay: float = 1e-4,
    gradient_clip_norm: float = 5.0,
    device: str = "cpu",
    prepared_train: PreparedDataset | None = None,
    prepared_validation: PreparedDataset | None = None,
    pairwise_state: PairwiseState | None = None,
) -> TrainingResult:
    resolved = _resolve_device(device)
    rng = set_seed(seed)
    model = LinearScorer().to(resolved)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    train_data = prepared_train or _materialize_dataset(train, resolved)
    validation_data = prepared_validation or _materialize_dataset(validation, resolved)
    if formulation == "pairwise" and pairwise_state is None:
        pairwise_state = _build_pairwise_state(
            formulation, train, validation, pair_delta,
            max_pairs_per_job, seed, resolved,
        )
    train_pair_tensors = pairwise_state.train_tensors if pairwise_state else None
    validation_pair_tensors = pairwise_state.validation_tensors if pairwise_state else None

    best_state = None
    best_ndcg = float("-inf")
    best_loss = float("inf")
    best_train_loss = float("inf")
    best_epoch = 0
    wait = 0
    final_epoch = 0
    history_rows = []
    for epoch in range(1, max_epochs + 1):
        final_epoch = epoch
        _run_training_epoch(
            formulation, model, optimizer, train_data, train_pair_tensors,
            batch_size, listwise_temperature, listwise_logit_epsilon,
            gradient_clip_norm, rng,
        )
        model.eval()
        with torch.no_grad():
            train_loss = _objective_loss(
                formulation, model, train_data, train_pair_tensors,
                listwise_temperature, listwise_logit_epsilon,
            )
            validation_loss = _objective_loss(
                formulation, model, validation_data, validation_pair_tensors,
                listwise_temperature, listwise_logit_epsilon,
            )
        validation_ndcg = _weak_validation_ndcg_at_5(model, validation_data)
        history_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            "generalization_gap": validation_loss - train_loss,
            "validation_weak_ndcg_at_5": validation_ndcg,
        })
        improved = validation_ndcg > best_ndcg + 1e-12
        tied_better_loss = (
            abs(validation_ndcg - best_ndcg) <= 1e-12
            and validation_loss < best_loss - 1e-6
        )
        if improved or tied_better_loss:
            best_ndcg = validation_ndcg
            best_loss = validation_loss
            best_train_loss = train_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is None:
        raise RuntimeError("Training produced no checkpoint")
    model.load_state_dict(best_state)
    return TrainingResult(
        formulation=formulation,
        model=model,
        learning_rate=float(learning_rate),
        batch_size=int(batch_size),
        best_epoch=best_epoch,
        best_validation_loss=best_loss,
        best_train_loss=best_train_loss,
        epochs_ran=final_epoch,
        stopped_early=final_epoch < max_epochs,
        weight_decay=float(weight_decay),
        gradient_clip_norm=float(gradient_clip_norm),
        selection_metric="validation_weak_ndcg_at_5",
        best_validation_ndcg_at_5=float(best_ndcg),
        batch_unit=_batch_unit(formulation),
        device=str(resolved),
        history=pd.DataFrame(history_rows),
        train_pair_hash=pairwise_state.train_hash if pairwise_state else None,
        validation_pair_hash=pairwise_state.validation_hash if pairwise_state else None,
    )


In [ ]:
def select_hyperparameters(
    formulation: str,
    train: pd.DataFrame,
    validation: pd.DataFrame,
    seed: int,
    learning_rates: list[float],
    batch_sizes: list[int],
    max_epochs: int,
    patience: int,
    pair_delta: float,
    max_pairs_per_job: int,
    listwise_temperature: float,
    listwise_logit_epsilon: float = 1e-4,
    weight_decay: float = 1e-4,
    gradient_clip_norm: float = 5.0,
    device: str = "cpu",
) -> tuple[TrainingResult, pd.DataFrame]:
    resolved = _resolve_device(device)
    prepared_train = _materialize_dataset(train, resolved)
    prepared_validation = _materialize_dataset(validation, resolved)
    pairwise_state = _build_pairwise_state(
        formulation, train, validation, pair_delta,
        max_pairs_per_job, seed, resolved,
    )
    results = []
    best = None
    for learning_rate in learning_rates:
        for batch_size in batch_sizes:
            result = _train_one(
                formulation, train, validation, seed, float(learning_rate), int(batch_size),
                max_epochs, patience, pair_delta, max_pairs_per_job,
                listwise_temperature, listwise_logit_epsilon,
                weight_decay, gradient_clip_norm, str(resolved),
                prepared_train, prepared_validation, pairwise_state,
            )
            results.append(result.metadata())
            if (
                best is None
                or result.best_validation_ndcg_at_5
                > best.best_validation_ndcg_at_5 + 1e-12
                or (
                    abs(
                        result.best_validation_ndcg_at_5
                        - best.best_validation_ndcg_at_5
                    ) <= 1e-12
                    and result.best_validation_loss < best.best_validation_loss
                )
            ):
                best = result
    return best, pd.DataFrame(results)


## 7. Metric, paired bootstrap và khóa Gold-test

nDCG@5, nDCG@10 và MRR được tính riêng cho từng job. MRR coi grade `≥2` là relevant. Paired bootstrap lấy mẫu lại các job, không coi từng cặp CV–job là quan sát độc lập.


In [ ]:
def dcg_at_k(relevance, k: int) -> float:
    values = np.asarray(relevance, dtype=float)[:k]
    if not len(values):
        return 0.0
    return float(np.sum((2.0 ** values - 1.0) / np.log2(np.arange(2, len(values) + 2))))


In [ ]:
def ndcg_at_k(relevance, scores, k: int) -> float:
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    actual = dcg_at_k(relevance[order], k)
    ideal = dcg_at_k(np.sort(relevance)[::-1], k)
    return 0.0 if ideal == 0.0 else float(actual / ideal)


In [ ]:
def reciprocal_rank(relevance, scores, threshold: int = 2) -> float:
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    positions = np.flatnonzero(relevance[order] >= threshold)
    return 0.0 if not len(positions) else float(1.0 / (positions[0] + 1))


In [ ]:
def per_job_metrics(
    frame: pd.DataFrame,
    scores,
    target_column: str = "relevance",
    k_values: tuple[int, ...] = (5, 10),
) -> pd.DataFrame:
    evaluation = frame[["job_id", target_column]].copy()
    evaluation["score"] = np.asarray(scores, dtype=float)
    rows = []
    for job_id, group in evaluation.groupby("job_id", sort=True):
        row = {"job_id": job_id}
        for k in k_values:
            row[f"ndcg@{k}"] = ndcg_at_k(group[target_column], group["score"], k)
        row["mrr"] = reciprocal_rank(group[target_column], group["score"], threshold=2)
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def macro_metrics(per_job: pd.DataFrame) -> dict:
    columns = [column for column in per_job.columns if column != "job_id"]
    return {column: float(per_job[column].mean()) for column in columns}


In [ ]:
def paired_job_bootstrap(
    per_job: pd.DataFrame,
    baseline: str,
    proposed: str,
    metric: str,
    n_resamples: int,
    seed: int,
) -> dict:
    subset = per_job[per_job["system"].isin([baseline, proposed])]
    pivot = subset.pivot(index="job_id", columns="system", values=metric)[[baseline, proposed]].dropna()
    if pivot.empty:
        raise ValueError("No paired jobs for bootstrap")
    differences = (pivot[proposed] - pivot[baseline]).to_numpy(float)
    rng = np.random.RandomState(seed)
    means = np.empty(n_resamples, dtype=float)
    for index in range(n_resamples):
        sample = rng.randint(0, len(differences), size=len(differences))
        means[index] = differences[sample].mean()
    return {
        "baseline": baseline,
        "proposed": proposed,
        "metric": metric,
        "n_jobs": int(len(differences)),
        "mean_delta": float(differences.mean()),
        "ci_95_low": float(np.percentile(means, 2.5)),
        "ci_95_high": float(np.percentile(means, 97.5)),
        "supports_improvement": bool(np.percentile(means, 2.5) > 0.0),
    }


In [ ]:
def select_formulation(
    validation_summary: pd.DataFrame,
    tie_tolerance: float = 0.005,
) -> str:
    required = {"formulation", "ndcg@5", "validation_pairwise_loss"}
    missing = required - set(validation_summary.columns)
    if missing:
        raise ValueError(
            "Formulation selection requires: " + ", ".join(sorted(missing))
        )
    means = validation_summary.groupby("formulation", as_index=False).agg(
        **{"ndcg@5": ("ndcg@5", "mean")},
        validation_pairwise_loss=("validation_pairwise_loss", "mean"),
    )
    best_value = float(means["ndcg@5"].max())
    tied = means.loc[means["ndcg@5"] >= best_value - tie_tolerance].copy()
    best_loss = float(tied["validation_pairwise_loss"].min())
    tied = set(tied.loc[
        tied["validation_pairwise_loss"] <= best_loss + 1e-12,
        "formulation",
    ])
    simplicity = ["pointwise", "pairwise", "listwise"]
    return next(name for name in simplicity if name in tied)


In [ ]:
def protocol_payload_sha256(payload: dict) -> str:
    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


In [ ]:
def create_immutable_run_directory(root: str | Path, protocol_sha256: str) -> Path:
    if len(protocol_sha256) != 64:
        raise ValueError("Protocol SHA-256 must contain 64 hexadecimal characters")

    root = Path(root)
    target = root / protocol_sha256
    if target.exists():
        run_number = 2
        while (root / f"{protocol_sha256}-run-{run_number:03d}").exists():
            run_number += 1
        target = root / f"{protocol_sha256}-run-{run_number:03d}"

    target.mkdir(parents=True, exist_ok=False)
    return target


In [ ]:
@dataclass
class ExperimentProtocol:
    """Durable guard that makes Gold-test a one-time post-lock operation."""

    sentinel_path: Path | None = None
    protocol_sha256: str | None = None
    selected_threshold: float | None = None
    selected_formulation: str | None = None
    lock_metadata: dict = field(default_factory=dict)
    locked: bool = False

    def __post_init__(self) -> None:
        if self.sentinel_path is not None:
            self.sentinel_path = Path(self.sentinel_path)

    @property
    def test_opened(self) -> bool:
        return bool(self.sentinel_path and self.sentinel_path.exists())

    def lock(self, threshold: float, formulation: str, metadata: dict) -> dict:
        if self.locked:
            raise RuntimeError("Protocol is already locked")
        if formulation not in {"pointwise", "pairwise", "listwise"}:
            raise ValueError("Unknown selected formulation")
        self.selected_threshold = float(threshold)
        self.selected_formulation = formulation
        self.lock_metadata = dict(metadata)
        self.locked = True
        return self.manifest()

    def open_gold_test_once(self) -> None:
        if not self.locked:
            raise RuntimeError("Gold-test cannot be opened before protocol lock")
        if self.sentinel_path is None or self.protocol_sha256 is None:
            raise RuntimeError("Durable Gold-test sentinel is not configured")
        self.sentinel_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "protocol_sha256": self.protocol_sha256,
            "opened_at_utc": datetime.now(timezone.utc).isoformat(),
            "state": "gold-test-opened",
        }
        try:
            with self.sentinel_path.open("x", encoding="utf-8") as handle:
                json.dump(payload, handle, indent=2, ensure_ascii=False)
        except FileExistsError as error:
            raise RuntimeError(
                "Gold-test has already been opened for this protocol"
            ) from error

    def manifest(self) -> dict:
        return {
            "locked": self.locked,
            "protocol_sha256": self.protocol_sha256,
            "selected_threshold": self.selected_threshold,
            "selected_formulation": self.selected_formulation,
            "metadata": self.lock_metadata,
            "test_opened": self.test_opened,
            "gold_test_sentinel": (
                str(self.sentinel_path) if self.sentinel_path is not None else None
            ),
        }


## 8. Bộ điều phối giao thức full

Lớp dưới đây thực thi state machine: audit → development → weak supervision → formulation selection → chuẩn bị toàn bộ checkpoint chính/ablation → protocol lock → durable one-time Gold-test → finalize. Full run dừng trước ranking nếu điều kiện tiên quyết về chất lượng label model không đạt; đây là hành vi fail-fast theo giao thức, không phải lỗi kỹ thuật.


In [ ]:
class ThreeSignalExperiment:
    """Stage-oriented runner used by the executable notebook."""

    def __init__(self, config: dict, root: str | Path):
        self.root = Path(root).resolve()
        self.config_path = self.root / "configs" / "experiment_3signal.yaml"
        self.config = copy.deepcopy(config)
        self.smoke = False
        self.settings = copy.deepcopy(self.config)
        self.device = _resolve_device(self.settings["models"]["device"])
        self.data_root = self._resolve(self.settings["data_dir"]).resolve()
        self.gold_path = self._resolve(self.settings["gold_path"]).resolve()
        annotation_value = self.settings["gold"].get("independent_annotations_path")
        self.independent_annotations_path = (
            self._resolve(annotation_value).resolve() if annotation_value else None
        )
        mode = "full"
        self.protocol_payload = self._build_protocol_payload(mode)
        self.protocol_sha256 = protocol_payload_sha256(self.protocol_payload)
        output_root = self._resolve(self.settings["output_dir"]) / mode
        self.output_dir = create_immutable_run_directory(
            output_root, self.protocol_sha256
        )
        for name in ["audit", "diagnostics", "tables", "predictions", "checkpoints"]:
            (self.output_dir / name).mkdir(exist_ok=True)
        self.protocol = ExperimentProtocol(
            self.output_dir / "audit" / "gold_test_opened.json",
            protocol_sha256=self.protocol_sha256,
        )
        self.environment_manifest = self._write_environment_manifest()
        self.models: dict[int, dict[str, object]] = {}
        self.ablation_models: dict[int, object] = {}
        self.stage = "initialized"

    def _resolve(self, value: str | Path) -> Path:
        path = Path(value)
        return path if path.is_absolute() else self.root / path

    def _write_environment_manifest(self) -> dict:
        packages = {}
        for distribution in [
            "numpy", "pandas", "scipy", "scikit-learn", "torch", "PyYAML",
            "sentence-transformers", "transformers", "matplotlib", "jupyter",
            "nbconvert", "ipykernel",
        ]:
            try:
                packages[distribution] = importlib.metadata.version(distribution)
            except importlib.metadata.PackageNotFoundError:
                packages[distribution] = None
        manifest = {
            "python": {
                "version": platform.python_version(),
                "implementation": platform.python_implementation(),
            },
            "platform": {
                "system": platform.system(),
                "release": platform.release(),
                "machine": platform.machine(),
            },
            "device_policy": {
                "requested": str(self.settings["models"]["device"]),
                "resolved": str(self.device),
                "cuda_available": bool(torch.cuda.is_available()),
                "cuda_runtime": torch.version.cuda,
                "cudnn_version": torch.backends.cudnn.version(),
                "gpu_name": (
                    torch.cuda.get_device_name(self.device)
                    if self.device.type == "cuda"
                    else None
                ),
            },
            "packages": packages,
        }
        write_json(
            self.output_dir / "audit" / "environment_manifest.json",
            manifest,
        )
        return manifest

    def _build_protocol_payload(self, mode: str) -> dict:
        input_manifest = build_input_manifest(self.data_root)
        return {
            "protocol_version": self.settings["protocol_version"],
            "mode": mode,
            "settings": self.settings,
            "gold_split_policy": "sorted-job-id-seeded-shuffle-v1",
            "feature_definition": "three-signal-multilingual-embedding-v4",
            "model_selection_policy": "weak-validation-ndcg5-v1",
            "weak_label_definition": "negative-global-positive-tail-v1",
            "posterior_threshold_policy": "fixed-0.5-v1",
            "training_device_policy": "configured-device-required-v1",
            "input_files": input_manifest,
            "input_hashes": {
                "jobs": input_manifest["jobs"]["sha256"],
                "candidates": input_manifest["candidates"]["sha256"],
                "gold": sha256_file(self.gold_path),
                "independent_annotations": (
                    sha256_file(self.independent_annotations_path)
                    if self.independent_annotations_path
                    and self.independent_annotations_path.is_file()
                    else None
                ),
                "config": sha256_file(self.config_path),
            },
        }

    def audit_data_and_gold(self) -> tuple[pd.DataFrame, dict]:
        self.raw = load_raw_data(self.data_root)
        self.gold = load_gold_with_identity_check(self.gold_path, self.raw)
        self.gold_manifest = make_gold_split_manifest(
            self.gold,
            n_validation_jobs=int(self.settings["gold"]["validation_jobs"]),
            seed=int(self.settings["gold"]["split_seed"]),
        )
        split_path = (
            self.root / "data" / "splits" /
            "gold_split_manifest_label_blind_v3.json"
        )
        if split_path.exists():
            existing = json.loads(split_path.read_text(encoding="utf-8"))
            if existing != self.gold_manifest:
                raise RuntimeError(
                    "Existing label-blind Gold split manifest differs from protocol"
                )
        else:
            write_json(split_path, self.gold_manifest)
        self.gold_validation = self.gold[self.gold["job_id"].isin(self.gold_manifest["validation_jobs"])].reset_index(drop=True)
        self._gold_test_private = self.gold[self.gold["job_id"].isin(self.gold_manifest["test_jobs"])].reset_index(drop=True)
        self.iaa_audit = inter_annotator_agreement(
            self.independent_annotations_path,
            self.gold,
        )
        self.stage = "gold_audited"
        write_json(self.output_dir / "audit" / "raw_data_audit.json", self.raw.audit)
        write_json(self.output_dir / "audit" / "gold_split_manifest.json", self.gold_manifest)
        write_json(
            self.output_dir / "audit" / "inter_annotator_agreement.json",
            self.iaa_audit,
        )
        return self.gold_validation.copy(), dict(self.raw.audit)

    def prepare_development_data(self) -> dict:
        if self.stage != "gold_audited":
            raise RuntimeError("Audit Gold before development preparation")
        # Exclude every annotated entity, not merely Gold-test, from weak supervision.
        sampled = sample_development_entities(
            self.raw,
            n_jobs=int(self.settings["sample"]["n_jobs"]),
            n_candidates=int(self.settings["sample"]["n_candidates"]),
            candidates_per_job=int(self.settings["sample"]["candidates_per_job"]),
            seed=int(self.settings["seed"]),
            excluded_job_ids=set(self.gold["job_id"]),
            excluded_candidate_ids=set(self.gold["cand_id"]),
        )
        train_pairs, validation_pairs, test_pairs, self.development_manifest = split_development_pairs(
            sampled.pairs,
            train_ratio=float(self.settings["split"]["train_ratio"]),
            validation_ratio=float(self.settings["split"]["validation_ratio"]),
            seed=int(self.settings["split"]["seed"]),
        )
        self.feature_pipeline = ThreeSignalFeaturePipeline(
            semantic_model_name=str(self.settings["features"]["semantic_model_name"]),
            semantic_batch_size=int(self.settings["features"]["semantic_batch_size"]),
            semantic_device=str(self.device),
            semantic_max_features=int(self.settings["features"]["semantic_max_features"]),
            role_max_features=int(self.settings["features"]["role_max_features"]),
        ).fit(train_pairs, sampled.jobs, sampled.candidates)
        self.train_raw_features = self.feature_pipeline.transform(train_pairs, sampled.jobs, sampled.candidates)
        self.validation_raw_features = self.feature_pipeline.transform(validation_pairs, sampled.jobs, sampled.candidates)
        self.development_test_raw_features = self.feature_pipeline.transform(test_pairs, sampled.jobs, sampled.candidates)
        self.gold_validation_raw_features = self.feature_pipeline.transform_gold(self.gold_validation, self.raw)
        self.train, self.fill_values = self.feature_pipeline.impute_for_models(self.train_raw_features)
        self.validation, _ = self.feature_pipeline.impute_for_models(self.validation_raw_features, self.fill_values)
        self.development_test, _ = self.feature_pipeline.impute_for_models(self.development_test_raw_features, self.fill_values)
        self.gold_validation_features, _ = self.feature_pipeline.impute_for_models(self.gold_validation_raw_features, self.fill_values)
        feature_manifest = self.feature_pipeline.manifest()
        all_gold_jobs = set(self.gold["job_id"])
        all_gold_candidates = set(self.gold["cand_id"])
        if set(feature_manifest["fit_job_ids"]) & all_gold_jobs:
            raise AssertionError("Annotated job leaked into feature fit")
        if set(feature_manifest["fit_candidate_ids"]) & all_gold_candidates:
            raise AssertionError("Annotated candidate leaked into feature fit")
        self.sampled = sampled
        self.stage = "development_prepared"
        write_json(self.output_dir / "audit" / "development_split_manifest.json", self.development_manifest)
        write_json(self.output_dir / "audit" / "feature_manifest.json", feature_manifest)
        write_json(self.output_dir / "audit" / "imputation_values.json", self.fill_values)
        return {
            "sampled_jobs": len(sampled.jobs),
            "sampled_candidates": len(sampled.candidates),
            "sampled_pairs": len(sampled.pairs),
            "train_pairs": len(self.train),
            "validation_pairs": len(self.validation),
            "development_test_pairs": len(self.development_test),
        }

    def fit_weak_supervision(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        if self.stage != "development_prepared":
            raise RuntimeError("Prepare development data before weak supervision")
        weak_config = self.settings["weak_supervision"]
        self.weak_pipeline = ThreeSourceWeakLabelPipeline(
            negative_percentile=float(weak_config["negative_percentile"]),
            positive_percentile=float(weak_config["positive_percentile"]),
            threshold_policy=str(weak_config["threshold_policy"]),
            label_model_config=copy.deepcopy(weak_config["label_model"]),
        )
        self.train_weak = self.weak_pipeline.fit_transform_train(self.train)
        parameters_before = self.weak_pipeline.parameters()
        self.validation_weak = self.weak_pipeline.transform(self.validation)
        self.development_test_weak = self.weak_pipeline.transform(self.development_test)
        self.gold_validation_weak = self.weak_pipeline.transform(self.gold_validation_features)
        if parameters_before != self.weak_pipeline.parameters():
            raise AssertionError("Held-out inference mutated weak supervision")
        lf_stats, lf_pairs = lf_diagnostics(self.train_weak)
        if not lf_pairs.empty and lf_pairs["spearman"].abs().max() >= float(weak_config["max_abs_spearman"]):
            raise RuntimeError("LF marginal-dependence diagnostic exceeded the predeclared threshold")
        gold_truth = (self.gold_validation_weak["relevance"].to_numpy(int) >= int(self.settings["gold"]["binary_threshold"])).astype(int)
        consensus_predictions = strict_three_of_three(self.gold_validation_weak)
        consensus_quality = binary_quality(gold_truth, consensus_predictions)
        self.posterior_threshold = float(weak_config["posterior_threshold"])
        threshold_table = pd.DataFrame([{
            "threshold": self.posterior_threshold,
            "policy": "fixed-0.5",
            "selection_data": "none",
        }])
        label_model_quality = binary_quality(
            gold_truth,
            (self.gold_validation_weak["y_prob"].to_numpy(float) >= self.posterior_threshold).astype(float),
        )
        label_condition_passed = bool(
            label_model_quality["recall"] > consensus_quality["recall"]
            and label_model_quality["precision"]
            >= consensus_quality["precision"] - 0.02
        )
        self.label_quality = pd.DataFrame([
            {"method": "strict_3_of_3", **consensus_quality},
            {
                "method": "dawid_skene", "threshold": self.posterior_threshold,
                **label_model_quality, "condition_passed": label_condition_passed,
            },
        ])
        # Persist audit before the confirmatory gate so a failed prerequisite
        # still leaves Table 4 and diagnostics on disk for the report.
        lf_stats.to_csv(self.output_dir / "diagnostics" / "lf_statistics.csv", index=False)
        lf_pairs.to_csv(self.output_dir / "diagnostics" / "lf_pair_diagnostics.csv", index=False)
        threshold_table.to_csv(self.output_dir / "diagnostics" / "posterior_threshold_selection.csv", index=False)
        self.label_quality.to_csv(self.output_dir / "tables" / "label_quality_gold_validation.csv", index=False)
        write_json(self.output_dir / "audit" / "weak_supervision_parameters.json", parameters_before)
        self.train_weak[["pair_id", "job_id", "cand_id", *LF_COLUMNS, "y_prob"]].to_csv(
            self.output_dir / "audit" / "train_weak_labels.csv", index=False
        )
        gold_validation_diagnostics = self.gold_validation_weak[
            ["pair_id", "job_id", "cand_id", "relevance", *LF_COLUMNS, "y_prob"]
        ].copy()
        gold_validation_diagnostics["binary_relevance"] = gold_truth
        gold_validation_diagnostics["dawid_skene_prediction"] = (
            gold_validation_diagnostics["y_prob"] >= self.posterior_threshold
        ).astype(int)
        gold_validation_diagnostics.to_csv(
            self.output_dir / "diagnostics" /
            "gold_validation_weak_label_diagnostics.csv",
            index=False,
        )
        self.label_gate_passed = label_condition_passed
        self.label_gate_reason = (
            None
            if label_condition_passed
            else (
                "Dawid-Skene did not achieve strictly higher recall than strict 3/3 "
                "while keeping precision within the predeclared 0.02 margin"
            )
        )
        self.stage = (
            "weak_supervision_fitted"
            if label_condition_passed or self.smoke
            else "confirmatory_blocked"
        )
        return self.label_quality.copy(), lf_stats, lf_pairs

    def train_and_select_formulation(self) -> pd.DataFrame:
        if self.stage != "weak_supervision_fitted":
            raise RuntimeError("Fit weak supervision before rankers")
        model_config = self.settings["models"]
        validation_rows = []
        selection_rows = []
        history_rows = []
        for seed_value in self.settings["seeds"]:
            seed = int(seed_value)
            self.models[seed] = {}
            for formulation in ["pointwise", "pairwise", "listwise"]:
                result, grid = select_hyperparameters(
                    formulation,
                    self.train_weak,
                    self.validation_weak,
                    seed,
                    model_config["learning_rates"],
                    model_config["batch_sizes"][formulation],
                    int(model_config["max_epochs"]),
                    int(model_config["patience"]),
                    float(model_config["pair_delta"]),
                    int(model_config["max_pairs_per_job"]),
                    float(model_config["listwise_temperature"]),
                    float(model_config["listwise_logit_epsilon"]),
                    float(model_config["weight_decay"]),
                    float(model_config["gradient_clip_norm"]),
                    str(self.device),
                )
                self.models[seed][formulation] = result
                for record in grid.to_dict("records"):
                    selection_rows.append({"seed": seed, **record})
                history = result.history.copy()
                history.insert(0, "seed", seed)
                history.insert(1, "formulation", formulation)
                history.insert(2, "learning_rate", result.learning_rate)
                history.insert(3, "batch_size", result.batch_size)
                history_rows.append(history)
                scores = predict_scores(result.model, self.gold_validation_weak)
                validation_pair_table = build_pair_table(
                    self.validation_weak,
                    float(model_config["pair_delta"]),
                    int(model_config["max_pairs_per_job"]),
                    seed,
                )
                preferred, nonpreferred = _pair_tensors(
                    self.validation_weak, validation_pair_table, str(self.device)
                )
                result.model.eval()
                with torch.no_grad():
                    validation_pairwise_loss = float(
                        pairwise_loss(result.model, preferred, nonpreferred).item()
                    )
                metrics = per_job_metrics(
                    self.gold_validation_weak,
                    scores,
                    target_column=self.settings["gold"]["relevance_column"],
                    k_values=tuple(self.settings["gold"]["k_values"]),
                )
                metrics.insert(0, "seed", seed)
                metrics.insert(1, "formulation", formulation)
                metrics.insert(2, "validation_pairwise_loss", validation_pairwise_loss)
                validation_rows.append(metrics)
                torch.save(
                    result.model.state_dict(),
                    self.output_dir / "checkpoints" / f"{formulation}_seed{seed}.pt",
                )
        self.formulation_validation = pd.concat(validation_rows, ignore_index=True)
        self.selected_formulation = select_formulation(
            self.formulation_validation,
            tie_tolerance=float(model_config["formulation_tie_tolerance"]),
        )
        self.formulation_summary = self.formulation_validation.groupby("formulation", as_index=False)[["ndcg@5", "ndcg@10", "mrr"]].mean()
        self.formulation_summary["selected"] = self.formulation_summary["formulation"] == self.selected_formulation
        self.training_history = pd.concat(history_rows, ignore_index=True)
        self.overfitting_diagnostics = pd.DataFrame(selection_rows)[[
            "seed", "formulation", "learning_rate", "batch_size", "batch_unit",
            "device", "best_epoch", "epochs_ran", "selection_metric",
            "best_validation_ndcg_at_5", "best_train_loss",
            "best_validation_loss", "generalization_gap", "stopped_early",
            "weight_decay", "gradient_clip_norm",
        ]]
        self.stage = "formulation_selected"
        self.formulation_validation.to_csv(self.output_dir / "diagnostics" / "formulation_validation_per_job.csv", index=False)
        self.formulation_summary.to_csv(self.output_dir / "tables" / "formulation_selection.csv", index=False)
        self.training_history.to_csv(self.output_dir / "diagnostics" / "training_history.csv", index=False)
        self.overfitting_diagnostics.to_csv(
            self.output_dir / "diagnostics" / "overfitting_diagnostics.csv", index=False
        )
        pd.DataFrame(selection_rows).to_csv(self.output_dir / "audit" / "model_selection_grid.csv", index=False)
        return self.formulation_summary.copy()

    def evaluate_development_test_once(self) -> pd.DataFrame:
        if self.stage != "formulation_selected":
            raise RuntimeError(
                "Development-test diagnostic requires completed formulation selection"
            )
        if hasattr(self, "development_test_diagnostic"):
            raise RuntimeError("Development-test diagnostic has already been evaluated")
        selected_before = self.selected_formulation
        checkpoint_ids_before = {
            seed: id(self.models[int(seed)][self.selected_formulation])
            for seed in self.settings["seeds"]
        }
        rows = []
        for seed_value in self.settings["seeds"]:
            seed = int(seed_value)
            result = self.models[seed][self.selected_formulation]
            scores = predict_scores(result.model, self.development_test_weak)
            metrics = per_job_metrics(
                self.development_test_weak,
                scores,
                target_column="y_prob",
                k_values=tuple(self.settings["gold"]["k_values"]),
            ).drop(columns="mrr")
            metrics.insert(0, "seed", seed)
            rows.append(metrics)
        self.development_test_per_job = pd.concat(rows, ignore_index=True)
        macro = self.development_test_per_job.groupby("seed", as_index=False)[
            ["ndcg@5", "ndcg@10"]
        ].mean()
        macro.insert(1, "formulation", self.selected_formulation)
        macro["selection_role"] = "diagnostic_only_after_selection"
        self.development_test_diagnostic = macro
        if self.selected_formulation != selected_before:
            raise AssertionError("Development-test diagnostic changed formulation")
        checkpoint_ids_after = {
            seed: id(self.models[int(seed)][self.selected_formulation])
            for seed in self.settings["seeds"]
        }
        if checkpoint_ids_after != checkpoint_ids_before:
            raise AssertionError("Development-test diagnostic changed selected checkpoints")
        self.development_test_per_job.to_csv(
            self.output_dir / "diagnostics" /
            "development_test_weak_ranking_per_job.csv",
            index=False,
        )
        self.development_test_diagnostic.to_csv(
            self.output_dir / "tables" /
            "development_test_weak_ranking_diagnostic.csv",
            index=False,
        )
        return self.development_test_diagnostic.copy()

    def _train_ablation_mean_signal(self, seed: int):
        train = self.train_weak.copy()
        validation = self.validation_weak.copy()
        train["y_prob"] = train[FEATURE_COLUMNS].mean(axis=1)
        validation["y_prob"] = validation[FEATURE_COLUMNS].mean(axis=1)
        result, _ = select_hyperparameters(
            self.selected_formulation,
            train,
            validation,
            seed,
            self.settings["models"]["learning_rates"],
            self.settings["models"]["batch_sizes"][self.selected_formulation],
            int(self.settings["models"]["max_epochs"]),
            int(self.settings["models"]["patience"]),
            float(self.settings["models"]["pair_delta"]),
            int(self.settings["models"]["max_pairs_per_job"]),
            float(self.settings["models"]["listwise_temperature"]),
            float(self.settings["models"]["listwise_logit_epsilon"]),
            float(self.settings["models"]["weight_decay"]),
            float(self.settings["models"]["gradient_clip_norm"]),
            str(self.device),
        )
        return result

    def prepare_confirmatory_checkpoints(self) -> dict:
        if self.stage != "formulation_selected":
            raise RuntimeError("Select formulation before preparing checkpoints")
        if not hasattr(self, "development_test_diagnostic"):
            raise RuntimeError(
                "Run the diagnostic-only development-test check before checkpoints"
            )
        main_metadata = {}
        ablation_metadata = {}
        for seed_value in self.settings["seeds"]:
            seed = int(seed_value)
            main_path = (
                self.output_dir / "checkpoints" /
                f"{self.selected_formulation}_seed{seed}.pt"
            )
            if not main_path.exists():
                raise RuntimeError(f"Missing selected-model checkpoint: {main_path}")
            main_metadata[str(seed)] = {
                "seed": seed,
                "formulation": self.selected_formulation,
                "path": str(main_path.relative_to(self.output_dir)),
                "sha256": sha256_file(main_path),
                "training": self.models[seed][self.selected_formulation].metadata(),
            }

            ablation = self._train_ablation_mean_signal(seed)
            self.ablation_models[seed] = ablation
            ablation_path = (
                self.output_dir / "checkpoints" /
                f"ablation_mean_signal_{self.selected_formulation}_seed{seed}.pt"
            )
            torch.save(ablation.model.state_dict(), ablation_path)
            ablation_metadata[str(seed)] = {
                "seed": seed,
                "formulation": self.selected_formulation,
                "path": str(ablation_path.relative_to(self.output_dir)),
                "sha256": sha256_file(ablation_path),
                "training": ablation.metadata(),
            }
        self.checkpoint_metadata = {
            "main": main_metadata,
            "ablation_mean_signal": ablation_metadata,
        }
        self.stage = "confirmatory_checkpoints_prepared"
        write_json(
            self.output_dir / "audit" / "confirmatory_checkpoints.json",
            self.checkpoint_metadata,
        )
        return copy.deepcopy(self.checkpoint_metadata)

    def _validate_checkpoint_metadata(self) -> None:
        metadata = getattr(self, "checkpoint_metadata", None)
        if not isinstance(metadata, dict) or not metadata.get("ablation_mean_signal"):
            raise RuntimeError(
                "Complete ablation checkpoint metadata is required before Gold-test"
            )
        expected_seeds = {str(int(value)) for value in self.settings["seeds"]}
        for family in ["main", "ablation_mean_signal"]:
            records = metadata.get(family)
            if not isinstance(records, dict) or set(records) != expected_seeds:
                raise RuntimeError(
                    f"Complete {family} checkpoint metadata is required before Gold-test"
                )
            for record in records.values():
                path = self.output_dir / record["path"]
                if not path.is_file() or sha256_file(path) != record.get("sha256"):
                    raise RuntimeError(
                        f"Locked {family} checkpoint is missing or hash-mismatched"
                    )

    def lock_protocol(self) -> dict:
        if self.stage != "confirmatory_checkpoints_prepared":
            raise RuntimeError("Prepare all confirmatory checkpoints before protocol lock")
        self._validate_checkpoint_metadata()
        split_path = (
            self.root / "data" / "splits" /
            "gold_split_manifest_label_blind_v3.json"
        )
        manifest = self.protocol.lock(
            self.posterior_threshold,
            self.selected_formulation,
            {
                "seeds": [int(value) for value in self.settings["seeds"]],
                "feature_manifest": self.feature_pipeline.manifest(),
                "weak_parameters": self.weak_pipeline.parameters(),
                "threshold_policy": "fixed-0.5",
                "training_device_policy": str(self.device),
                "gold_validation_uses": [
                    "label_quality_at_fixed_threshold",
                    "formulation_selection",
                ],
                "development_test": {
                    "selection_role": "diagnostic_only_after_selection",
                    "summary": self.development_test_diagnostic.to_dict("records"),
                },
                "confirmatory_checkpoints": self.checkpoint_metadata,
                "environment_manifest": self.environment_manifest,
                "environment_manifest_sha256": sha256_file(
                    self.output_dir / "audit" / "environment_manifest.json"
                ),
                "gold_validation_jobs": self.gold_manifest["validation_jobs"],
                "gold_test_jobs_sha256": sha256_file(split_path),
                "protocol_payload": self.protocol_payload,
            },
        )
        self.stage = "protocol_locked"
        write_json(self.output_dir / "audit" / "protocol_lock.json", manifest)
        return manifest

    def evaluate_gold_test_once(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        if self.stage != "protocol_locked":
            raise RuntimeError("Protocol must be locked before Gold-test")
        self._validate_checkpoint_metadata()
        locked_checkpoints = self.protocol.lock_metadata.get(
            "confirmatory_checkpoints"
        )
        if locked_checkpoints != self.checkpoint_metadata:
            raise RuntimeError(
                "Protocol lock does not contain the current ablation checkpoint metadata"
            )
        self.protocol.open_gold_test_once()
        # Gold-test is transformed and weak-labeled only after the one-time gate opens.
        gold_test_raw = self.feature_pipeline.transform_gold(self._gold_test_private, self.raw)
        gold_test_features, _ = self.feature_pipeline.impute_for_models(gold_test_raw, self.fill_values)
        parameters_before_test = self.weak_pipeline.parameters()
        test = self.weak_pipeline.transform(gold_test_features)
        if parameters_before_test != self.weak_pipeline.parameters():
            raise AssertionError("Gold-test inference mutated weak supervision")
        per_job_rows = []
        prediction_rows = []
        for seed_value in self.settings["seeds"]:
            seed = int(seed_value)
            selected_model = copy.deepcopy(
                self.models[seed][self.selected_formulation].model
            ).to(self.device)
            selected_record = self.checkpoint_metadata["main"][str(seed)]
            selected_model.load_state_dict(torch.load(
                self.output_dir / selected_record["path"],
                map_location=self.device,
                weights_only=True,
            ))
            ablation_model = copy.deepcopy(
                self.ablation_models[seed].model
            ).to(self.device)
            ablation_record = self.checkpoint_metadata[
                "ablation_mean_signal"
            ][str(seed)]
            ablation_model.load_state_dict(torch.load(
                self.output_dir / ablation_record["path"],
                map_location=self.device,
                weights_only=True,
            ))
            systems = {
                "manual_score_h": test["heuristic_score"].to_numpy(float),
                "selected_ltr": predict_scores(selected_model, test),
                "ablation_mean_signal_ltr": predict_scores(ablation_model, test),
                "ablation_direct_probability": test["y_prob"].to_numpy(float),
            }
            for system, scores in systems.items():
                metrics = per_job_metrics(
                    test,
                    scores,
                    target_column=self.settings["gold"]["relevance_column"],
                    k_values=tuple(self.settings["gold"]["k_values"]),
                )
                metrics.insert(0, "seed", seed)
                metrics.insert(1, "system", system)
                per_job_rows.append(metrics)
                prediction_rows.append(pd.DataFrame({
                    "seed": seed,
                    "system": system,
                    "job_id": test["job_id"].to_numpy(),
                    "cand_id": test["cand_id"].to_numpy(),
                    "score": np.asarray(scores, dtype=float),
                }))
        self.gold_test_per_job = pd.concat(per_job_rows, ignore_index=True)
        averaged_per_job = self.gold_test_per_job.groupby(["system", "job_id"], as_index=False)[["ndcg@5", "ndcg@10", "mrr"]].mean()
        self.gold_test_summary = averaged_per_job.groupby("system", as_index=False)[["ndcg@5", "ndcg@10", "mrr"]].mean()
        comparisons = [
            ("manual_score_h", "selected_ltr", "main"),
            ("ablation_mean_signal_ltr", "selected_ltr", "core_1"),
            ("ablation_direct_probability", "selected_ltr", "core_2"),
        ]
        bootstrap_rows = []
        for baseline, proposed, comparison in comparisons:
            for metric in ["ndcg@5", "ndcg@10", "mrr"]:
                bootstrap_rows.append({
                    "comparison": comparison,
                    **paired_job_bootstrap(
                        averaged_per_job,
                        baseline,
                        proposed,
                        metric,
                        int(self.settings["bootstrap"]["n_resamples"]),
                        int(self.settings["bootstrap"]["seed"]),
                    ),
                })
        self.bootstrap_results = pd.DataFrame(bootstrap_rows)
        self.stage = "gold_test_evaluated"
        self.gold_test_per_job.to_csv(self.output_dir / "diagnostics" / "gold_test_per_job_by_seed.csv", index=False)
        self.gold_test_summary.to_csv(self.output_dir / "tables" / "gold_test_main_and_ablations.csv", index=False)
        self.bootstrap_results.to_csv(self.output_dir / "tables" / "paired_bootstrap_ci.csv", index=False)
        pd.concat(prediction_rows, ignore_index=True).to_csv(self.output_dir / "predictions" / "gold_test_predictions.csv", index=False)
        write_json(self.output_dir / "audit" / "protocol_final.json", self.protocol.manifest())
        return self.gold_test_summary.copy(), self.bootstrap_results.copy(), averaged_per_job

    def finalize(self) -> dict:
        if self.stage != "gold_test_evaluated":
            raise RuntimeError("Complete Gold-test evaluation before finalization")
        main = self.bootstrap_results[
            (self.bootstrap_results["comparison"] == "main")
            & (self.bootstrap_results["metric"] == "ndcg@5")
        ].iloc[0]
        conclusion = (
            "SUPPORTED: selected three-signal LTR improves nDCG@5 over the manual score."
            if bool(main["supports_improvement"])
            else "NOT SUPPORTED: the nDCG@5 confidence interval contains or touches zero."
        )
        manifest = {
            "protocol_version": self.settings["protocol_version"],
            "mode": "full",
            "stage": "complete",
            "selected_formulation": self.selected_formulation,
            "selected_posterior_threshold": self.posterior_threshold,
            "conclusion": conclusion,
            "counts": {
                "gold_validation_jobs": len(self.gold_manifest["validation_jobs"]),
                "gold_test_jobs": len(self.gold_manifest["test_jobs"]),
                "seeds": len(self.settings["seeds"]),
            },
            "input_files": build_input_manifest(self.data_root),
            "input_hashes": {
                "jobs": sha256_file(self.data_root / "JOB_DATA_FINAL.csv"),
                "candidates": sha256_file(self.data_root / "USER_DATA_FINAL.csv"),
                "gold": sha256_file(self.gold_path),
            },
        }
        write_json(self.output_dir / "audit" / "run_manifest.json", manifest)
        return manifest


## 9. Nguồn dữ liệu thực nghiệm thật và kiểm toán đầu vào

Thực nghiệm này chỉ đọc hai bảng raw tại `data/` tương đối từ project root và Gold benchmark đã khóa. Không có synthetic data, fixture hoặc fallback. Mỗi file được ghi absolute path đã resolve, dung lượng, thời điểm sửa và SHA-256 trước mọi phép lấy mẫu hay feature engineering.


In [ ]:
input_manifest = build_input_manifest(DATA_ROOT)
raw_preview = load_raw_data(DATA_ROOT)
gold_preview = load_gold_with_identity_check(GOLD_PATH, raw_preview)

display(pd.DataFrame(input_manifest).T.reset_index(names="input"))
display(pd.DataFrame({
    "dataset": ["Jobs raw", "Jobs clean", "CV raw", "CV clean", "Gold"],
    "rows": [
        raw_preview.audit["jobs_raw_rows"],
        raw_preview.audit["jobs_clean_rows"],
        raw_preview.audit["candidates_raw_rows"],
        raw_preview.audit["candidates_clean_rows"],
        len(gold_preview),
    ],
    "columns": [
        raw_preview.audit["jobs_columns"],
        raw_preview.jobs.shape[1] - 1,
        raw_preview.audit["candidates_columns"],
        raw_preview.candidates.shape[1] - 1,
        gold_preview.shape[1],
    ],
    "exact_duplicates_removed": [
        raw_preview.audit["jobs_exact_duplicates"], 0,
        raw_preview.audit["candidates_exact_duplicates"], 0, 0,
    ],
    "missing_cells_raw": [
        raw_preview.audit["jobs_missing_cells"], np.nan,
        raw_preview.audit["candidates_missing_cells"], np.nan, np.nan,
    ],
}))
display(gold_preview["relevance"].value_counts().sort_index().rename_axis("grade").to_frame("count"))

assert raw_preview.audit["data_root"] == str(DATA_ROOT)
assert raw_preview.audit["jobs_raw_rows"] == 14634
assert raw_preview.audit["jobs_clean_rows"] == 14634
assert raw_preview.audit["candidates_raw_rows"] == 3983
assert raw_preview.audit["candidates_exact_duplicates"] == 792
assert raw_preview.audit["candidates_clean_rows"] == 3191
assert len(gold_preview) == 100
assert gold_preview["job_id"].nunique() == 12
assert gold_preview["cand_id"].nunique() == 67


## 10. Audit identity và cố định Gold split

Runner đọc lại đúng các file đã băm ở trên, kiểm tra identity giữa Gold và dữ liệu gốc, grade hợp lệ và tính query-disjoint của split. Chưa có feature hoặc score Gold-test nào được tạo tại bước này.


In [ ]:
experiment = ThreeSignalExperiment(CONFIG, ROOT)
gold_validation, raw_audit = experiment.audit_data_and_gold()

assert raw_audit["input_files"] == input_manifest
assert raw_audit["data_root"] == str(DATA_ROOT)
display(pd.DataFrame([{
    key: value for key, value in raw_audit.items()
    if key not in {"input_files", "jobs_required_columns", "candidates_required_columns"}
}]).T.rename(columns={0: "value"}))
display(pd.DataFrame({
    "partition": ["Gold-validation", "Gold-test"],
    "jobs": [len(experiment.gold_manifest["validation_jobs"]), len(experiment.gold_manifest["test_jobs"])],
    "pairs": [experiment.gold_manifest["validation_pairs"], experiment.gold_manifest["test_pairs"]],
}))
display(experiment.gold["relevance"].value_counts().sort_index().rename_axis("grade").to_frame("count"))

assert set(experiment.gold_manifest["validation_jobs"]).isdisjoint(experiment.gold_manifest["test_jobs"])
assert len(experiment.gold_manifest["validation_jobs"]) == CONFIG["gold"]["validation_jobs"]
assert len(experiment.gold_manifest["test_jobs"]) == experiment.gold["job_id"].nunique() - CONFIG["gold"]["validation_jobs"]
display(pd.DataFrame([experiment.iaa_audit]))


## 11. Development pool thật, query-disjoint split và train-only preprocessing

Toàn bộ job và CV đã xuất hiện trong Gold đều bị loại khỏi development pool. Feature pipeline được fit duy nhất trên development-train; cùng trạng thái đó được dùng cho development-validation và Gold-validation.


In [ ]:
development_counts = experiment.prepare_development_data()
feature_manifest = experiment.feature_pipeline.manifest()

assert len(experiment.sampled.jobs) == 2_000
assert len(experiment.sampled.candidates) == 1_500
assert len(experiment.sampled.pairs) == 400_000
assert not experiment.sampled.pairs.duplicated(["job_id", "cand_id"]).any()
assert set(experiment.sampled.pairs["job_id"]).isdisjoint(set(experiment.gold["job_id"]))
assert set(experiment.sampled.pairs["cand_id"]).isdisjoint(set(experiment.gold["cand_id"]))
assert set(experiment.development_manifest["train_jobs"]).isdisjoint(
    experiment.development_manifest["validation_jobs"]
)
assert set(experiment.development_manifest["train_jobs"]).isdisjoint(
    experiment.development_manifest["test_jobs"]
)
assert set(experiment.development_manifest["validation_jobs"]).isdisjoint(
    experiment.development_manifest["test_jobs"]
)
assert set(experiment.sampled.pairs["job_source_index"]).issubset(
    set(raw_preview.jobs["_source_index"])
)
assert set(experiment.sampled.pairs["candidate_source_index"]).issubset(
    set(raw_preview.candidates["_source_index"])
)

display(pd.DataFrame([development_counts]).T.rename(columns={0: "count"}))
display(pd.DataFrame({
    "partition": ["train", "validation", "development-test"],
    "jobs": [
        len(experiment.development_manifest["train_jobs"]),
        len(experiment.development_manifest["validation_jobs"]),
        len(experiment.development_manifest["test_jobs"]),
    ],
    "pairs": [len(experiment.train), len(experiment.validation), len(experiment.development_test)],
}))
display(pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "train_missing_rate": [experiment.train_raw_features[c].isna().mean() for c in FEATURE_COLUMNS],
    "train_mean_after_imputation": [experiment.train[c].mean() for c in FEATURE_COLUMNS],
    "train_std_after_imputation": [experiment.train[c].std() for c in FEATURE_COLUMNS],
}))
display({
    "semantic_encoder": feature_manifest["semantic_encoder"],
    "role_lexical_vocabulary_size": feature_manifest["role_lexical_vocabulary_size"],
    "description_lexical_vocabulary_size": feature_manifest["description_lexical_vocabulary_size"],
    "feature_columns": feature_manifest["feature_columns"],
})

assert feature_manifest["feature_columns"] == FEATURE_COLUMNS
assert set(feature_manifest["fit_job_ids"]).isdisjoint(set(experiment.gold["job_id"]))
assert set(feature_manifest["fit_candidate_ids"]).isdisjoint(set(experiment.gold["cand_id"]))


## 11. Chất lượng nguồn nhãn trên Gold-validation

Đây là điều kiện tiên quyết của Core 1. Ngưỡng âm của mỗi LF lấy phân vị 25 trên development-train; để tránh sparse skill overlap bị suy biến do nhiều giá trị 0 trùng nhau, ngưỡng dương lấy phân vị 75 trên phần train strictly-above-negative. Các ngưỡng được đóng băng trước held-out inference. Luật strict 3/3 là baseline có abstention; Dawid–Skene được đánh giá tại threshold cố định 0,5, không calibration trên Gold-validation. Nếu điều kiện đã khai báo trước không đạt, full confirmatory run chủ động dừng và **không mở Gold-test**.


In [ ]:
label_quality, lf_statistics, lf_pair_diagnostics = (
    experiment.fit_weak_supervision()
)

display(Markdown("### Thống kê labeling functions trên development-train"))
display(lf_statistics)
display(Markdown("### Phụ thuộc giữa các labeling functions"))
display(lf_pair_diagnostics)
display(Markdown("### Bảng chất lượng nhãn trên Gold-validation"))
display(label_quality)
print("Selected posterior threshold:", experiment.posterior_threshold)
print("Weak-supervision artifacts:", experiment.output_dir)

CONFIRMATORY_ALLOWED = bool(experiment.label_gate_passed)
if CONFIRMATORY_ALLOWED:
    display(Markdown(
        "**PASS:** label-model prerequisite đạt; các bước confirmatory được phép chạy."
    ))
else:
    strict_row = label_quality.loc[
        label_quality["method"] == "strict_3_of_3"
    ].iloc[0]
    label_row = label_quality.loc[
        label_quality["method"] == "dawid_skene"
    ].iloc[0]
    display(Markdown(
        "**BLOCKED:** label-model prerequisite không đạt. "
        f"Gold-validation có `{int(label_row['n_positive'])}` positive; "
        f"strict 3/3: TP={int(strict_row['tp'])}, FP={int(strict_row['fp'])}, "
        f"FN={int(strict_row['fn'])}, TN={int(strict_row['tn'])}; "
        f"Dawid–Skene: TP={int(label_row['tp'])}, FP={int(label_row['fp'])}, "
        f"FN={int(label_row['fn'])}, TN={int(label_row['tn'])}. "
        "Ranking confirmatory và Gold-test sẽ được bỏ qua; đây là kết quả "
        "fail-fast hợp lệ, không phải lỗi thực thi."
    ))


## 12. Huấn luyện đa seed và lựa chọn formulation

Mỗi formulation được huấn luyện với năm seed và cùng hyperparameter grid. Hyperparameter được chọn bằng objective trên development-validation; formulation được chọn bằng macro nDCG@5 trên bốn Gold-validation jobs. Không có Gold-test score tại bước này.


In [ ]:
if CONFIRMATORY_ALLOWED:
    formulation_summary = experiment.train_and_select_formulation()
    display(formulation_summary.sort_values(
        "ndcg@5", ascending=False
    ).reset_index(drop=True))
    display(Markdown("### Chẩn đoán overfitting tại checkpoint tốt nhất"))
    display(experiment.overfitting_diagnostics.sort_values(
        ["formulation", "seed", "best_validation_loss"]
    ).reset_index(drop=True))
    print("Selected formulation:", experiment.selected_formulation)
else:
    formulation_summary = pd.DataFrame()
    display(Markdown(
        "**SKIPPED:** formulation training bị chặn bởi label-model gate."
    ))


### 12.1 Final weak-ranking internal check

Development-test chỉ được dùng **sau khi** formulation và checkpoint đã được chọn. Kết quả này là diagnostic-only, không được dùng để thay đổi bất kỳ quyết định huấn luyện hay protocol nào.


In [ ]:
if CONFIRMATORY_ALLOWED:
    selected_before_development_test = experiment.selected_formulation
    development_test_diagnostic = experiment.evaluate_development_test_once()
    display(development_test_diagnostic)
    assert experiment.selected_formulation == selected_before_development_test
    assert set(development_test_diagnostic["selection_role"]) == {
        "diagnostic_only_after_selection"
    }
else:
    development_test_diagnostic = pd.DataFrame()
    display(Markdown(
        "**SKIPPED:** development-test diagnostic bị chặn bởi label-model gate."
    ))


### 12.2 Learning curves và generalization gap

Đường train/validation objective được ghi ở mọi epoch cho checkpoint hyperparameter được chọn. Early stopping dùng macro weak-validation nDCG@5; Gold không tham gia quyết định dừng. Objective loss và khoảng cách validation–train là diagnostic, không phải tiêu chí chọn checkpoint.


In [ ]:
if CONFIRMATORY_ALLOWED:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
    for axis, formulation in zip(
        axes, ["pointwise", "pairwise", "listwise"]
    ):
        subset = experiment.training_history[
            experiment.training_history["formulation"] == formulation
        ]
        mean_curve = subset.groupby("epoch", as_index=False)[
            ["train_loss", "validation_loss"]
        ].mean()
        axis.plot(
            mean_curve["epoch"], mean_curve["train_loss"], label="train"
        )
        axis.plot(
            mean_curve["epoch"], mean_curve["validation_loss"],
            label="validation"
        )
        axis.set_title(formulation)
        axis.set_xlabel("epoch")
        axis.set_ylabel("objective loss")
        axis.grid(alpha=0.25)
        axis.legend()
    plt.suptitle("Mean learning curves across five seeds")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown(
        "**SKIPPED:** không có learning curves vì ranker không được huấn luyện."
    ))


## 13. Chuẩn bị checkpoint và Protocol lock

Toàn bộ checkpoint của mô hình chính và ablation mean-signal được train, lưu và băm **trước** khi khóa. Threshold posterior cố định 0,5, formulation, seed, feature vocabulary hashes, tham số Dawid–Skene, development-test diagnostic-only, checkpoint metadata và hash Gold split được ghi ra trước khi Gold-test được transform. Cell assertion xác nhận test vẫn đóng tại thời điểm khóa.


In [ ]:
if CONFIRMATORY_ALLOWED:
    checkpoint_metadata = experiment.prepare_confirmatory_checkpoints()
    assert set(checkpoint_metadata) == {"main", "ablation_mean_signal"}
    protocol_lock = experiment.lock_protocol()
    display(protocol_lock)
    assert protocol_lock["locked"] is True
    assert protocol_lock["test_opened"] is False
else:
    checkpoint_metadata = {}
    protocol_lock = {
        "locked": False,
        "test_opened": False,
        "status": "confirmatory_blocked",
        "reason": experiment.label_gate_reason,
    }
    display(protocol_lock)


## 14. Đánh giá Gold-test đúng một lần

Cell này là điểm duy nhất mở Gold-test. Bốn hệ thống được báo cáo:

1. `manual_score_h`: công thức thủ công lịch sử.
2. `selected_ltr`: hệ thống đầy đủ.
3. `ablation_mean_signal_ltr`: thay Dawid–Skene bằng trung bình ba tín hiệu, giữ formulation.
4. `ablation_direct_probability`: bỏ LTR, xếp hạng trực tiếp bằng posterior Dawid–Skene.

Mỗi score được đánh giá theo job, trung bình qua seed trước khi bootstrap, rồi macro-average qua job.


In [ ]:
if CONFIRMATORY_ALLOWED:
    gold_test_summary, bootstrap_results, gold_test_per_job = (
        experiment.evaluate_gold_test_once()
    )
    display(Markdown("### Kết quả chính và đúng hai ablation"))
    display(gold_test_summary.sort_values(
        "ndcg@5", ascending=False
    ).reset_index(drop=True))
    display(Markdown("### Paired bootstrap 95% CI theo job"))
    display(bootstrap_results)
    assert set(gold_test_summary["system"]) == {
        "manual_score_h", "selected_ltr",
        "ablation_mean_signal_ltr", "ablation_direct_probability",
    }
    assert set(bootstrap_results["comparison"]) == {
        "main", "core_1", "core_2"
    }
else:
    gold_test_summary = pd.DataFrame()
    bootstrap_results = pd.DataFrame()
    gold_test_per_job = pd.DataFrame()
    display(Markdown(
        "**NOT OPENED:** Gold-test vẫn được giữ kín vì label-model gate thất bại."
    ))
    assert experiment.protocol.test_opened is False


## 15. Kết luận xác nhận và artifact

Giả thuyết chính chỉ được ủng hộ khi cận dưới CI 95% của chênh lệch nDCG@5 (`selected_ltr − manual_score_h`) lớn hơn 0. Kết luận được sinh bằng quy tắc cố định, không lựa chọn diễn giải sau khi xem kết quả.


In [ ]:
if CONFIRMATORY_ALLOWED:
    run_manifest = experiment.finalize()
    main_ndcg5 = bootstrap_results.query(
        "comparison == 'main' and metric == 'ndcg@5'"
    ).iloc[0]
    display(Markdown(f"## {run_manifest['conclusion']}"))
    display(pd.DataFrame([main_ndcg5]))
    print("Authoritative artifacts:", experiment.output_dir)
    print("Protocol final state:", experiment.protocol.manifest())
else:
    run_manifest = {
        "conclusion": "CONFIRMATORY EXPERIMENT BLOCKED",
        "reason": experiment.label_gate_reason,
        "gold_test_opened": False,
        "artifact_directory": str(experiment.output_dir),
    }
    write_json(
        experiment.output_dir / "audit" / "blocked_run_manifest.json",
        run_manifest,
    )
    display(Markdown("## CONFIRMATORY EXPERIMENT BLOCKED"))
    display(pd.DataFrame([run_manifest]))
    print("Diagnostic artifacts:", experiment.output_dir)
    assert experiment.protocol.test_opened is False


## 16. Giới hạn diễn giải

- Gold hiện chỉ gồm 12 query và được gán nhãn bởi một annotator; IAA phải báo `not_available` cho đến khi có file nhãn độc lập thật. Độ bất định có thể lớn và benchmark không đại diện cho quyết định tuyển dụng thực tế.
- Dataset không có click, apply, interview hoặc hiring outcome.
- Dawid–Skene giả định độc lập có điều kiện, trong khi embedding text có thể chứa thông tin về skill và experience; tương quan LF được báo cáo như diagnostic chứ không chứng minh giả định đúng.
- Ba tín hiệu là proxy quan sát được. Posterior của label model chỉ là xác suất ước lượng dùng để huấn luyện, không phải ground truth.
- Với Gold hiện tại, Gold-test chỉ có tám job; paired-bootstrap CI phải được báo cáo cùng point estimate và không suy rộng từ từng cặp CV–job như các quan sát độc lập.
- Nếu điều kiện chất lượng label model thất bại ở Mục 11, nghiên cứu không được phép bỏ qua gate, điều chỉnh theo Gold-test hoặc diễn giải kết quả ranking xác nhận.
